<a href="https://colab.research.google.com/github/Junnihub/Costas_Hub/blob/main/CoastHub.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Descargar CoastSat directamente al entorno de Google Colab
!git clone https://github.com/kvos/CoastSat.git
%cd CoastSat

# Instalar las librerías necesarias
!pip install geopandas earthengine-api scikit-image matplotlib astropy

In [ ]:
# --- CÉLULA DE RESET TOTAL PARA GITHUB ---
# 1. Borra todas las salidas (outputs) de todas las celdas
from google.colab import output
output.clear()

# 2. Borra todas las variables almacenadas en memoria (limpieza de kernel)
# Esto es vital para que al guardar no se guarde el "estado" de tus datos
%reset -f

print("Estado del notebook reseteado. El notebook ahora está limpio de datos y salidas.")

In [ ]:
!pip -q install shapely
!pip -q install shapely pandas

In [ ]:
import ee
# iniciar sesión para darle permiso a Colab de usar Earth Engine
ee.Authenticate()
ee.Initialize(project='costas-491722') #Entre comillas, colocar el nombre de tu projecto de GEE, pues costas-491722 está vinculado a mi cuenta personal

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**INICIO PRUEBAS**

In [ ]:
"""###Coordenadas ejemplo a periodos actuales

import os
import glob
import shutil
import matplotlib
import matplotlib.pyplot as plt
matplotlib.use('Agg') # Fuerza a matplotlib a no usar ventanas gráficas

# --- PARCHE PARA GOOGLE COLAB ---
# Engañamos a matplotlib para que cuando CoastSat llame a "showMaximized()"
# no haga absolutamente nada, evitando el error de 'window'.
def dummy_maximize():
    pass

class DummyWindow:
    def showMaximized(self):
        pass

class DummyManager:
    window = DummyWindow()

# Reemplazamos la función que da el error
plt.get_current_fig_manager = lambda: DummyManager()
# --------------------------------

from IPython.display import Image, display
from coastsat import SDS_download, SDS_preprocess, SDS_shoreline

# 1. CONFIGURACIÓN DEL EJEMPLO (Narrabeen)
sitename = 'NARRA_ejemplo'
filepath = os.path.join(os.getcwd(), 'data')
carpeta = os.path.join(filepath, sitename)

if os.path.exists(carpeta):
    shutil.rmtree(carpeta)

polygon = [
    [[-74.9712061448,10.9801614263],[-74.9730179403,10.9827378702],[-74.9576595169,10.9931460152],[-74.9558477214,10.9905696621],[-74.9712061448,10.9801614263]]
]

dates = ['2025-09-01', '2025-10-31']
sat_list = ['S2']

inputs = {
    'polygon': polygon,
    'dates': dates,
    'sat_list': sat_list,
    'sitename': sitename,
    'filepath': filepath
}

# 2. DESCARGA Y PREPROCESAMIENTO
print("Descargando...")
metadata = SDS_download.retrieve_images(inputs)

settings = {
    'cloud_thresh': 0.5,
    'dist_clouds': 300,
    'output_epsg': 28356,
    'check_detection': False,
    'adjust_detection': False,
    'save_figure': True,        # Clave para generar el mapa coloreado
    's2cloudless_prob': 60,
    'sand_color': 'default',
    'cloud_mask_issue': False,
    'pan_off': False,
    'min_beach_area': 1000,
    'min_length_sl': 200,
    'inputs': inputs
}

print("Preprocesando imágenes...")
SDS_preprocess.save_jpg(metadata, settings, use_matplotlib=True)

# 3. EXTRACCIÓN (Con el parche activado, ahora no colapsará)
print("Extrayendo líneas de costa y generando mapas de clasificación...")
output = SDS_shoreline.extract_shorelines(metadata, settings)

# 4. BUSCAR Y MOSTRAR RESULTADOS
carpeta_figuras = os.path.join(filepath, sitename, 'jpg_files', 'detection')

if os.path.exists(carpeta_figuras):
    imagenes_color = sorted(glob.glob(os.path.join(carpeta_figuras, '*.jpg')))

    if len(imagenes_color) > 0:
        print("\n¡Éxito! Aquí está la imagen clasificada con agua (azul), arena (naranja) y la línea de costa:")
        ultima_img_color = imagenes_color[-1]
        display(Image(filename=ultima_img_color))

        try:
            from google.colab import files
            files.download(ultima_img_color)
        except:
            pass
    else:
        print("No se encontraron archivos JPG en la carpeta de detección.")
else:
    print(f"No se encontró la carpeta de salida: {carpeta_figuras}")
"""

APARTIR DE ACÁ ACABAN LOS CÓDIGOS DE PRUEBA, SE EXTRAEN LOS QUE SE VAN A UTILZAR

In [ ]:
"""###
import os
import glob
import shutil
import numpy as np
import imageio.v2 as imageio
import matplotlib
import matplotlib.pyplot as plt
matplotlib.use('Agg')
from IPython.display import Image, display
from coastsat import SDS_download, SDS_preprocess, SDS_shoreline

# 1. NOMBRE DEL SEGMENTO
sitename = 'Puerto_Colombia_3_Miramar'
filepath = os.path.join(os.getcwd(), 'data')
carpeta = os.path.join(filepath, sitename)

if os.path.exists(carpeta):
    shutil.rmtree(carpeta)

# 1. NOMBRE DEL SEGMENTO
sitename = 'Puerto_Colombia_3_Miramar'

###



# 2. COORDENADAS
polygon = [
    [[-74.9549848389,10.9974806231],[-74.9566216486,10.9979285027],[-74.9540448839,11.0070026237],[-74.9524080742,11.0065547579],[-74.9549848389,10.9974806231]]
]

dates = ['2026-04-29', '2026-06-03']
sat_list = ['S2', 'L8']
inputs = {'polygon': polygon, 'dates': dates, 'sat_list': sat_list, 'sitename': sitename, 'filepath': filepath}


# 3. DESCARGA
print(f"1. Descargando imágenes para el segmento: {sitename}...")
metadata = SDS_download.retrieve_images(inputs)

# 4. CONFIGURACIÓN MEJORADA (Anti-Espagueti)

# 4. CONFIGURACIÓN MEJORADA (Anti-Espagueti)
settings = {
    'cloud_thresh': 0.1,
    'dist_clouds': 300,
    'output_epsg': 32618,
    'check_detection': False,
    'adjust_detection': False,
    'save_figure': True,
    's2cloudless_prob': 60,
    'sand_color': 'default',
    'cloud_mask_issue': False,
    'pan_off': False,
    'min_beach_area': 1000,
    'min_length_sl': 1000,
    'inputs': inputs
}

# 5. PREPROCESAMIENTO Y EXTRACCIÓN
print("2. Preprocesando imágenes y filtrando ruido...")
SDS_preprocess.save_jpg(metadata, settings, use_matplotlib=True)

def dummy_maximize(): pass
class DummyWindow:
    def showMaximized(self): pass
class DummyManager: window = DummyWindow()
plt.get_current_fig_manager = lambda: DummyManager()

print("3. Ejecutando extracción de línea de costa...")
output = SDS_shoreline.extract_shorelines(metadata, settings)

from google.colab import files

# --- GENERACIÓN DE RESULTADOS ---
print("\n--- GENERANDO RESULTADOS ---")

# A. IMAGEN CLASIFICADA
carpeta_deteccion = os.path.join(filepath, sitename, 'jpg_files', 'detection')
if os.path.exists(carpeta_deteccion):
    imagenes_color = sorted(glob.glob(os.path.join(carpeta_deteccion, '*.jpg')))
    if len(imagenes_color) > 0:
        ultima_img = imagenes_color[-1]
        print("\n✓ 1. Imagen clasificada lista:")
        display(Image(filename=ultima_img))

# B. GRÁFICO HISTÓRICO (MAPEO COHERENTE)
if len(output['shorelines']) > 0:
    print("\n✓ 2. Generando gráfico histórico coherente (sin líneas cruzadas)...")
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.axis('equal')
    ax.set_xlabel('Eastings (Metros) [EPSG:32618]', fontsize=12)
    ax.set_ylabel('Northings (Metros) [EPSG:32618]', fontsize=12)
    ax.set_title(f'Evolución de la Línea de Costa - {sitename}', fontsize=14)
    ax.grid(linestyle='--', color='gray', alpha=0.7)

    colores = plt.cm.jet(np.linspace(0, 1, len(output['shorelines'])))
    fechas_graficadas = 0
    fechas_usadas = [] # Para que no se repitan fechas en la leyenda

    for i in range(len(output['shorelines'])):
        sl = output['shorelines'][i]
        date = output['dates'][i]

        # Validar que no esté vacía
        if len(sl) > 0:
            fecha_str = date.strftime('%d-%m-%Y')
            etiqueta = fecha_str if fecha_str not in fechas_usadas else None
            if etiqueta: fechas_usadas.append(fecha_str)

            # ¡TRUCO 2! Cambiamos '-' por '.' (Puntos) y quitamos el conectador
            ax.plot(sl[:, 0], sl[:, 1], '.', color=colores[i], markersize=4, label=etiqueta)
            fechas_graficadas += 1

    if fechas_graficadas > 0:
        ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), title="Fechas", fontsize=8, markerscale=3)
        plt.tight_layout()
        ruta_grafico = os.path.join(filepath, sitename, f'{sitename}_Mapeo.png')
        plt.savefig(ruta_grafico, dpi=300, bbox_inches='tight')
        plt.close(fig)
        display(Image(filename=ruta_grafico))
        try: files.download(ruta_grafico)
        except: pass

# C. VIDEO TIMELAPSE CON CLASIFICACIÓN (AZUL/NARANJA)
print("\n✓ 3. Creando Video Timelapse Clasificado (.mp4)...")
ruta_video = os.path.join(filepath, sitename, f"{sitename}_timelapse_clasificado.mp4")

if os.path.exists(carpeta_deteccion):
    images_list = sorted(glob.glob(os.path.join(carpeta_deteccion, '*.jpg')))
    if len(images_list) > 0:
        try:
            frames = [imageio.imread(img_path) for img_path in images_list]
            imageio.mimsave(ruta_video, frames, fps=3)
            print(f"¡Video creado con éxito! Iniciando descarga...")
            try: files.download(ruta_video)
            except: pass
        except Exception as e:
            print(f"Error creando el video: {e}")

print("\n¡ANÁLISIS COMPLETADO AL 100%!")

import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import LineString
from shapely.ops import nearest_points

def to_linestring(sl):
    if sl is None or len(sl) < 2:
        return None
    pts = [(float(x), float(y)) for x, y in sl]
    if len(set(pts)) < 2:
        return None
    return LineString(pts)

def unit(v):
    v = np.asarray(v, dtype=float)
    n = np.linalg.norm(v)
    return v / n if n > 0 else v

def make_transect(p, L, half_len=300, eps=10.0):
    s = L.project(p)
    s1 = max(0, s - eps)
    s2 = min(L.length, s + eps)
    p1 = L.interpolate(s1)
    p2 = L.interpolate(s2)
    t = unit([p2.x - p1.x, p2.y - p1.y])      # tangente
    n = np.array([-t[1], t[0]])               # normal
    a = (p.x - half_len*n[0], p.y - half_len*n[1])
    b = (p.x + half_len*n[0], p.y + half_len*n[1])
    return LineString([a, b]), n

def intersect_signed(transect, normal_vec, ref_point, shoreline):
    inter = transect.intersection(shoreline)
    if inter.is_empty:
        return None, None

    # elegir punto de intersección más cercano al ref_point
    if inter.geom_type == "Point":
        q = inter
    elif inter.geom_type == "MultiPoint":
        q = sorted(list(inter.geoms), key=lambda g: g.distance(ref_point))[0]
    else:
        # casos raros: LineString/GeometryCollection -> punto más cercano
        q = nearest_points(ref_point, inter)[1]

    vec = np.array([q.x - ref_point.x, q.y - ref_point.y])
    signed = float(np.dot(vec, normal_vec))
    return signed, q

# --- 0) Selección de costas inicial/final ---
idx_validos = [i for i, sl in enumerate(output['shorelines']) if sl is not None and len(sl) >= 50]
if len(idx_validos) < 2:
    raise ValueError("No hay suficientes shorelines válidas para generar informe.")

i0, i1 = idx_validos[0], idx_validos[-1]
date0, date1 = output['dates'][i0], output['dates'][i1]
L0, L1 = to_linestring(output['shorelines'][i0]), to_linestring(output['shorelines'][i1])
if L0 is None or L1 is None:
    raise ValueError("No se pudo construir LineString para costa inicial/final.")

# --- 1) Crear puntos de medición (transectos) a lo largo de la costa inicial ---
spacing_m = 50            # separación entre transectos (ajústalo)
transect_half = 300       # longitud hacia cada lado (ajústalo)
eps_tangent = 10.0

n = max(8, int(L0.length // spacing_m))
dist_along = np.linspace(0.05*L0.length, 0.95*L0.length, n)
ref_points = [L0.interpolate(d) for d in dist_along]

rows = []
for k, p in enumerate(ref_points, start=1):
    tr, nvec = make_transect(p, L0, half_len=transect_half, eps=eps_tangent)
    d0, q0 = intersect_signed(tr, nvec, p, L0)
    d1, q1 = intersect_signed(tr, nvec, p, L1)
    if d0 is None or d1 is None:
        continue

    delta = d1 - d0  # metros (firmado)

    rows.append({
        "transect_id": k,
        "ref_x": float(p.x),
        "ref_y": float(p.y),
        "delta_m": float(delta),
        "start_x": float(q0.x), "start_y": float(q0.y),
        "end_x": float(q1.x), "end_y": float(q1.y),
    })

df = pd.DataFrame(rows)
if df.empty or len(df) < 5:
    raise ValueError("Muy pocos transectos válidos. Ajusta el recuadro o parámetros.")

# --- 2) Filtrado robusto (MAD) para quitar escenarios atípicos ---
d = df["delta_m"].to_numpy()
med = float(np.median(d))
mad = float(np.median(np.abs(d - med)) + 1e-9)
z = 0.6745 * (d - med) / mad
df["robust_z"] = z
df["is_outlier"] = np.abs(z) > 3.5

df_ok = df[~df["is_outlier"]].copy()

def summary(x):
    x = np.asarray(x, dtype=float)
    return {
        "n": int(len(x)),
        "mean_m": float(np.mean(x)),
        "median_m": float(np.median(x)),
        "p10_m": float(np.percentile(x, 10)),
        "p90_m": float(np.percentile(x, 90)),
        "min_m": float(np.min(x)),
        "max_m": float(np.max(x)),
    }

summary_raw = summary(df["delta_m"])
summary_ok  = summary(df_ok["delta_m"])

# --- 3) Guardar POLÍGONO analizado (lat/lon) tal como lo usaste ---
polygon_latlon = inputs["polygon"][0]  # lista [ [lon,lat], ... ]

# --- 4) Exportar archivos (para tu “base de datos” web) ---
out_dir = os.path.join(filepath, sitename)
os.makedirs(out_dir, exist_ok=True)

csv_path  = os.path.join(out_dir, f"{sitename}_transects_report.csv")
json_path = os.path.join(out_dir, f"{sitename}_report.json")
txt_path  = os.path.join(out_dir, f"{sitename}_report.txt")
png_path  = os.path.join(out_dir, f"{sitename}_report_map.png")

df.to_csv(csv_path, index=False)

report = {
    "sitename": sitename,
    "epsg": 32618,
    "dates": {
        "start": date0.strftime("%Y-%m-%d"),
        "end": date1.strftime("%Y-%m-%d"),
    },
    "polygon_latlon": polygon_latlon,
    "transect_spacing_m": spacing_m,
    "transect_half_length_m": transect_half,
    "summary_raw": summary_raw,
    "summary_filtered": summary_ok,
    "notes": [
        "delta_m es el desplazamiento a lo largo del transecto entre fecha final e inicial.",
        "El signo depende de la orientación de la normal del transecto; para reportes públicos, puede mostrarse también |delta_m|.",
        "is_outlier se marca con un filtro robusto MAD para remover detecciones atípicas.",
    ],
    "transects": df.to_dict(orient="records"),
}
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

# TXT humano
with open(txt_path, "w", encoding="utf-8") as f:
    f.write(f"Informe de cambio de línea de costa\n")
    f.write(f"Segmento: {sitename}\n")
    f.write(f"EPSG: 32618 (metros)\n")
    f.write(f"Periodo: {date0.strftime('%Y-%m-%d')} → {date1.strftime('%Y-%m-%d')}\n\n")
    f.write("Polígono analizado (lon,lat):\n")
    for pt in polygon_latlon:
        f.write(f"  - {pt}\n")
    f.write("\nResumen (sin filtrar):\n")
    f.write(json.dumps(summary_raw, ensure_ascii=False, indent=2) + "\n")
    f.write("\nResumen (filtrado robusto MAD):\n")
    f.write(json.dumps(summary_ok, ensure_ascii=False, indent=2) + "\n")
    f.write("\nDetalle por transecto: ver CSV/JSON.\n")

# --- 5) PNG tipo “mapita”: costa inicio vs fin + transectos ---
plt.figure(figsize=(8,8))
x0, y0 = np.array(output["shorelines"][i0])[:,0], np.array(output["shorelines"][i0])[:,1]
x1, y1 = np.array(output["shorelines"][i1])[:,0], np.array(output["shorelines"][i1])[:,1]
plt.plot(x0, y0, ".", ms=1, label=f"Inicio {date0.strftime('%Y-%m-%d')}")
plt.plot(x1, y1, ".", ms=1, label=f"Fin {date1.strftime('%Y-%m-%d')}")

# dibujar segmentos start->end de cada transecto
for _, r in df_ok.iterrows():
    plt.plot([r["start_x"], r["end_x"]], [r["start_y"], r["end_y"]], "-", lw=1, alpha=0.7)

plt.gca().set_aspect("equal", "box")
plt.grid(ls="--", alpha=0.4)
plt.title(f"{sitename} | Cambio de costa (transectos)\nMediana: {summary_ok['median_m']:.2f} m")
plt.xlabel("Easting (m) [EPSG:32618]")
plt.ylabel("Northing (m) [EPSG:32618]")
plt.legend()
plt.tight_layout()
plt.savefig(png_path, dpi=250)
plt.close()

print("=== Informe generado ===")
print("CSV :", csv_path)
print("JSON:", json_path)
print("TXT :", txt_path)
print("PNG :", png_path)

"""###

**EXTRACCIÓN TRANSECTOS FINALES** - En este apartado se recomienda únicamente cambiar las fechas para la actualización de la BD




---
**1**

---









In [ ]:
import os
import glob
import shutil
import numpy as np
import pandas as pd
import imageio.v2 as imageio
import matplotlib
import matplotlib.pyplot as plt
matplotlib.use('Agg')
from IPython.display import Image, display
from shapely.geometry import LineString
from google.colab import files

# Módulos oficiales de CoastSat
from coastsat import SDS_download, SDS_preprocess, SDS_shoreline, SDS_transects, SDS_tools

# 1. NOMBRE DEL SEGMENTO Y CARPETA
sitename = 'segmento_malecon_general'
filepath = os.path.join(os.getcwd(), 'data')
carpeta = os.path.join(filepath, sitename)

if os.path.exists(carpeta):
    shutil.rmtree(carpeta)

# 2. COORDENADAS
polygon = [
    [[-74.955228809,10.9934844961],[-74.9595260928,10.9946603747],[-74.9556923608,11.008160976],[-74.951395077,11.0069851513],[-74.955228809,10.9934844961]]
]

dates = ['2026-04-29', '2026-06-03']
sat_list = ['S2', 'L8']
inputs = {'polygon': polygon, 'dates': dates, 'sat_list': sat_list, 'sitename': sitename, 'filepath': filepath}

# 3. DESCARGA
print(f"1. Descargando imágenes para el segmento: {sitename}...")
metadata = SDS_download.retrieve_images(inputs)

# 4. CONFIGURACIÓN
settings = {
    'cloud_thresh': 0.1,
    'dist_clouds': 300,
    'output_epsg': 32618,
    'check_detection': False,
    'adjust_detection': False,
    'save_figure': True,
    's2cloudless_prob': 60,
    'sand_color': 'default',
    'cloud_mask_issue': False,
    'pan_off': False,
    'min_beach_area': 1000,
    'min_length_sl': 800,  # Bajado a 800m para asegurar que encaje en tu recuadro
    'inputs': inputs
}

# 5. PREPROCESAMIENTO Y EXTRACCIÓN
print("2. Preprocesando imágenes y filtrando ruido...")
SDS_preprocess.save_jpg(metadata, settings, use_matplotlib=True)

def dummy_maximize(): pass
class DummyWindow:
    def showMaximized(self): pass
class DummyManager: window = DummyWindow()
plt.get_current_fig_manager = lambda: DummyManager()

print("3. Ejecutando extracción de línea de costa...")
output = SDS_shoreline.extract_shorelines(metadata, settings)


# --- GENERACIÓN DE RESULTADOS VISUALES ---
print("\n--- GENERANDO RESULTADOS VISUALES ---")

# A. IMAGEN CLASIFICADA
carpeta_deteccion = os.path.join(filepath, sitename, 'jpg_files', 'detection')
if os.path.exists(carpeta_deteccion):
    imagenes_color = sorted(glob.glob(os.path.join(carpeta_deteccion, '*.jpg')))
    if len(imagenes_color) > 0:
        ultima_img = imagenes_color[-1]
        print("\n✓ 1. Imagen clasificada lista:")
        display(Image(filename=ultima_img))

# B. GRÁFICO HISTÓRICO (MAPEO COHERENTE)
if len(output['shorelines']) > 0:
    print("\n✓ 2. Generando gráfico histórico coherente...")
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.axis('equal')
    ax.set_xlabel('Eastings (Metros) [EPSG:32618]', fontsize=12)
    ax.set_ylabel('Northings (Metros) [EPSG:32618]', fontsize=12)
    ax.set_title(f'Evolución de la Línea de Costa - {sitename}', fontsize=14)
    ax.grid(linestyle='--', color='gray', alpha=0.7)

    colores = plt.cm.jet(np.linspace(0, 1, len(output['shorelines'])))
    fechas_usadas = []

    for i in range(len(output['shorelines'])):
        sl = output['shorelines'][i]
        date = output['dates'][i]
        if len(sl) > 0:
            fecha_str = date.strftime('%d-%m-%Y')
            etiqueta = fecha_str if fecha_str not in fechas_usadas else None
            if etiqueta: fechas_usadas.append(fecha_str)
            ax.plot(sl[:, 0], sl[:, 1], '.', color=colores[i], markersize=4, label=etiqueta)

    if len(fechas_usadas) > 0:
        ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), title="Fechas", fontsize=8, markerscale=3)
        plt.tight_layout()
        ruta_grafico = os.path.join(filepath, sitename, f'{sitename}_Mapeo.png')
        plt.savefig(ruta_grafico, dpi=300, bbox_inches='tight')
        plt.close(fig)
        display(Image(filename=ruta_grafico))

# C. VIDEO TIMELAPSE
print("\n✓ 3. Creando Video Timelapse Clasificado (.mp4)...")
ruta_video = os.path.join(filepath, sitename, f"{sitename}_timelapse_clasificado.mp4")
if os.path.exists(carpeta_deteccion):
    images_list = sorted(glob.glob(os.path.join(carpeta_deteccion, '*.jpg')))
    if len(images_list) > 0:
        try:
            frames = [imageio.imread(img_path) for img_path in images_list]
            imageio.mimsave(ruta_video, frames, fps=3)
            print(f"¡Video creado con éxito!")
        except Exception as e:
            print(f"Error creando el video: {e}")


# --- INFORME ROBUSTO NATIVO DE COASTSAT ---
print("\n--- GENERANDO INFORME ANALÍTICO (TRANSECTOS NATIVOS) ---")

if len(output['shorelines']) > 0:
    # Seleccionar la primera línea de costa válida para trazar transectos
    idx_validos = [i for i, sl in enumerate(output['shorelines']) if len(sl) > 50]

    if len(idx_validos) > 0:
        L0 = LineString(output['shorelines'][idx_validos[0]])
        transects = dict([])
        espaciado_m = 100       # Un transecto cada 100m
        longitud_mitad = 300    # Largo del transecto (300m al mar, 300m a tierra)

        # Generación automática de transectos
        for i, dist in enumerate(np.arange(10, L0.length, espaciado_m)):
            p = L0.interpolate(dist)
            p_antes = L0.interpolate(max(0, dist - 5))
            p_despues = L0.interpolate(min(L0.length, dist + 5))
            t = np.array([p_despues.x - p_antes.x, p_despues.y - p_antes.y])
            t = t / np.linalg.norm(t) if np.linalg.norm(t) > 0 else np.array([1,0])
            n = np.array([-t[1], t[0]])
            p1 = np.array([p.x, p.y]) - n * longitud_mitad
            p2 = np.array([p.x, p.y]) + n * longitud_mitad
            transects[f'Transecto_{i+1}'] = np.array([p1, p2])

        # Control de Calidad Nativo (QC)
        settings_transects = {
            'along_dist': 25,
            'min_points': 3,
            'max_std': 15,
            'max_range': 30,
            'min_chainage': -100,
            'multiple_inter': 'auto',
            'auto_prc': 0.1,
        }

        # Calcular intersecciones con CoastSat
        cross_distance = SDS_transects.compute_intersection_QC(output, transects, settings_transects)

        # Exportar a CSV tabular
        out_dict = {'dates': output['dates']}
        for key in transects.keys():
            out_dict[key] = cross_distance[key]

        df = pd.DataFrame(out_dict)
        ruta_csv = os.path.join(filepath, sitename, f'{sitename}_reporte_robusto.csv')
        df.to_csv(ruta_csv, sep=',', index=False)
        print(f"✓ Informe CSV creado en: {ruta_csv}")

        # Exportar a GeoJSON
        gdf = SDS_tools.transects_to_gdf(transects)
        gdf.set_crs(epsg=settings['output_epsg'], inplace=True, allow_override=True)
        ruta_geojson = os.path.join(filepath, sitename, f'{sitename}_transectos.geojson')
        gdf.to_file(ruta_geojson, driver='GeoJSON', encoding='utf-8')
        print(f"✓ Archivo GeoJSON de transectos creado en: {ruta_geojson}")

print("\n¡ANÁLISIS COMPLETADO AL 100%!")



---
**2**

---




In [ ]:
import os
import glob
import shutil
import numpy as np
import pandas as pd
import imageio.v2 as imageio
import matplotlib
import matplotlib.pyplot as plt
matplotlib.use('Agg')
from IPython.display import Image, display
from shapely.geometry import LineString
from google.colab import files

# Módulos oficiales de CoastSat
from coastsat import SDS_download, SDS_preprocess, SDS_shoreline, SDS_transects, SDS_tools

# 1. NOMBRE DEL SEGMENTO Y CARPETA
sitename = 'segmento_puerto_salgar_amplio'
filepath = os.path.join(os.getcwd(), 'data')
carpeta = os.path.join(filepath, sitename)

if os.path.exists(carpeta):
    shutil.rmtree(carpeta)

# 2. COORDENADAS
polygon = [
    [[-74.9609813684,10.9859768141],[-74.9667253842,10.9899172795],[-74.9571417187,11.0033794658],[-74.9513977029,10.9994391803],[-74.9609813684,10.9859768141]]
]

dates = ['2026-04-29', '2026-06-03']
sat_list = ['S2', 'L8']
inputs = {'polygon': polygon, 'dates': dates, 'sat_list': sat_list, 'sitename': sitename, 'filepath': filepath}

# 3. DESCARGA
print(f"1. Descargando imágenes para el segmento: {sitename}...")
metadata = SDS_download.retrieve_images(inputs)

# 4. CONFIGURACIÓN
settings = {
    'cloud_thresh': 0.1,
    'dist_clouds': 300,
    'output_epsg': 32618,
    'check_detection': False,
    'adjust_detection': False,
    'save_figure': True,
    's2cloudless_prob': 60,
    'sand_color': 'default',
    'cloud_mask_issue': False,
    'pan_off': False,
    'min_beach_area': 1000,
    'min_length_sl': 800,  # Bajado a 800m para asegurar que encaje en tu recuadro
    'inputs': inputs
}

# 5. PREPROCESAMIENTO Y EXTRACCIÓN
print("2. Preprocesando imágenes y filtrando ruido...")
SDS_preprocess.save_jpg(metadata, settings, use_matplotlib=True)

def dummy_maximize(): pass
class DummyWindow:
    def showMaximized(self): pass
class DummyManager: window = DummyWindow()
plt.get_current_fig_manager = lambda: DummyManager()

print("3. Ejecutando extracción de línea de costa...")
output = SDS_shoreline.extract_shorelines(metadata, settings)


# --- GENERACIÓN DE RESULTADOS VISUALES ---
print("\n--- GENERANDO RESULTADOS VISUALES ---")

# A. IMAGEN CLASIFICADA
carpeta_deteccion = os.path.join(filepath, sitename, 'jpg_files', 'detection')
if os.path.exists(carpeta_deteccion):
    imagenes_color = sorted(glob.glob(os.path.join(carpeta_deteccion, '*.jpg')))
    if len(imagenes_color) > 0:
        ultima_img = imagenes_color[-1]
        print("\n✓ 1. Imagen clasificada lista:")
        display(Image(filename=ultima_img))

# B. GRÁFICO HISTÓRICO (MAPEO COHERENTE)
if len(output['shorelines']) > 0:
    print("\n✓ 2. Generando gráfico histórico coherente...")
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.axis('equal')
    ax.set_xlabel('Eastings (Metros) [EPSG:32618]', fontsize=12)
    ax.set_ylabel('Northings (Metros) [EPSG:32618]', fontsize=12)
    ax.set_title(f'Evolución de la Línea de Costa - {sitename}', fontsize=14)
    ax.grid(linestyle='--', color='gray', alpha=0.7)

    colores = plt.cm.jet(np.linspace(0, 1, len(output['shorelines'])))
    fechas_usadas = []

    for i in range(len(output['shorelines'])):
        sl = output['shorelines'][i]
        date = output['dates'][i]
        if len(sl) > 0:
            fecha_str = date.strftime('%d-%m-%Y')
            etiqueta = fecha_str if fecha_str not in fechas_usadas else None
            if etiqueta: fechas_usadas.append(fecha_str)
            ax.plot(sl[:, 0], sl[:, 1], '.', color=colores[i], markersize=4, label=etiqueta)

    if len(fechas_usadas) > 0:
        ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), title="Fechas", fontsize=8, markerscale=3)
        plt.tight_layout()
        ruta_grafico = os.path.join(filepath, sitename, f'{sitename}_Mapeo.png')
        plt.savefig(ruta_grafico, dpi=300, bbox_inches='tight')
        plt.close(fig)
        display(Image(filename=ruta_grafico))

# C. VIDEO TIMELAPSE
print("\n✓ 3. Creando Video Timelapse Clasificado (.mp4)...")
ruta_video = os.path.join(filepath, sitename, f"{sitename}_timelapse_clasificado.mp4")
if os.path.exists(carpeta_deteccion):
    images_list = sorted(glob.glob(os.path.join(carpeta_deteccion, '*.jpg')))
    if len(images_list) > 0:
        try:
            frames = [imageio.imread(img_path) for img_path in images_list]
            imageio.mimsave(ruta_video, frames, fps=3)
            print(f"¡Video creado con éxito!")
        except Exception as e:
            print(f"Error creando el video: {e}")


# --- INFORME ROBUSTO NATIVO DE COASTSAT ---
print("\n--- GENERANDO INFORME ANALÍTICO (TRANSECTOS NATIVOS) ---")

if len(output['shorelines']) > 0:
    # Seleccionar la primera línea de costa válida para trazar transectos
    idx_validos = [i for i, sl in enumerate(output['shorelines']) if len(sl) > 50]

    if len(idx_validos) > 0:
        L0 = LineString(output['shorelines'][idx_validos[0]])
        transects = dict([])
        espaciado_m = 100       # Un transecto cada 100m
        longitud_mitad = 300    # Largo del transecto (300m al mar, 300m a tierra)

        # Generación automática de transectos
        for i, dist in enumerate(np.arange(10, L0.length, espaciado_m)):
            p = L0.interpolate(dist)
            p_antes = L0.interpolate(max(0, dist - 5))
            p_despues = L0.interpolate(min(L0.length, dist + 5))
            t = np.array([p_despues.x - p_antes.x, p_despues.y - p_antes.y])
            t = t / np.linalg.norm(t) if np.linalg.norm(t) > 0 else np.array([1,0])
            n = np.array([-t[1], t[0]])
            p1 = np.array([p.x, p.y]) - n * longitud_mitad
            p2 = np.array([p.x, p.y]) + n * longitud_mitad
            transects[f'Transecto_{i+1}'] = np.array([p1, p2])

        # Control de Calidad Nativo (QC)
        settings_transects = {
            'along_dist': 25,
            'min_points': 3,
            'max_std': 15,
            'max_range': 30,
            'min_chainage': -100,
            'multiple_inter': 'auto',
            'auto_prc': 0.1,
        }

        # Calcular intersecciones con CoastSat
        cross_distance = SDS_transects.compute_intersection_QC(output, transects, settings_transects)

        # Exportar a CSV tabular
        out_dict = {'dates': output['dates']}
        for key in transects.keys():
            out_dict[key] = cross_distance[key]

        df = pd.DataFrame(out_dict)
        ruta_csv = os.path.join(filepath, sitename, f'{sitename}_reporte_robusto.csv')
        df.to_csv(ruta_csv, sep=',', index=False)
        print(f"✓ Informe CSV creado en: {ruta_csv}")

        # Exportar a GeoJSON
        gdf = SDS_tools.transects_to_gdf(transects)
        gdf.set_crs(epsg=settings['output_epsg'], inplace=True, allow_override=True)
        ruta_geojson = os.path.join(filepath, sitename, f'{sitename}_transectos.geojson')
        gdf.to_file(ruta_geojson, driver='GeoJSON', encoding='utf-8')
        print(f"✓ Archivo GeoJSON de transectos creado en: {ruta_geojson}")

print("\n¡ANÁLISIS COMPLETADO AL 100%!")



---


**3**

---



In [ ]:
import os
import glob
import shutil
import numpy as np
import pandas as pd
import imageio.v2 as imageio
import matplotlib
import matplotlib.pyplot as plt
matplotlib.use('Agg')
from IPython.display import Image, display
from shapely.geometry import LineString
from google.colab import files

# Módulos oficiales de CoastSat
from coastsat import SDS_download, SDS_preprocess, SDS_shoreline, SDS_transects, SDS_tools

# 1. NOMBRE DEL SEGMENTO Y CARPETA
sitename = 'segmento_costa_oeste_puerto'
filepath = os.path.join(os.getcwd(), 'data')
carpeta = os.path.join(filepath, sitename)

if os.path.exists(carpeta):
    shutil.rmtree(carpeta)

# 2. COORDENADAS
polygon = [
    [[-74.9795909596,10.9750222025],[-74.9830137512,10.9808450194],[-74.9628076066,10.992291611],[-74.959384815,10.9864690198],[-74.9795909596,10.9750222025]]
]

dates = ['2026-04-29', '2026-06-03']
sat_list = ['S2', 'L8']
inputs = {'polygon': polygon, 'dates': dates, 'sat_list': sat_list, 'sitename': sitename, 'filepath': filepath}

# 3. DESCARGA
print(f"1. Descargando imágenes para el segmento: {sitename}...")
metadata = SDS_download.retrieve_images(inputs)

# 4. CONFIGURACIÓN
settings = {
    'cloud_thresh': 0.1,
    'dist_clouds': 300,
    'output_epsg': 32618,
    'check_detection': False,
    'adjust_detection': False,
    'save_figure': True,
    's2cloudless_prob': 60,
    'sand_color': 'default',
    'cloud_mask_issue': False,
    'pan_off': False,
    'min_beach_area': 1000,
    'min_length_sl': 800,  # Bajado a 800m para asegurar que encaje en tu recuadro
    'inputs': inputs
}

# 5. PREPROCESAMIENTO Y EXTRACCIÓN
print("2. Preprocesando imágenes y filtrando ruido...")
SDS_preprocess.save_jpg(metadata, settings, use_matplotlib=True)

def dummy_maximize(): pass
class DummyWindow:
    def showMaximized(self): pass
class DummyManager: window = DummyWindow()
plt.get_current_fig_manager = lambda: DummyManager()

print("3. Ejecutando extracción de línea de costa...")
output = SDS_shoreline.extract_shorelines(metadata, settings)


# --- GENERACIÓN DE RESULTADOS VISUALES ---
print("\n--- GENERANDO RESULTADOS VISUALES ---")

# A. IMAGEN CLASIFICADA
carpeta_deteccion = os.path.join(filepath, sitename, 'jpg_files', 'detection')
if os.path.exists(carpeta_deteccion):
    imagenes_color = sorted(glob.glob(os.path.join(carpeta_deteccion, '*.jpg')))
    if len(imagenes_color) > 0:
        ultima_img = imagenes_color[-1]
        print("\n✓ 1. Imagen clasificada lista:")
        display(Image(filename=ultima_img))

# B. GRÁFICO HISTÓRICO (MAPEO COHERENTE)
if len(output['shorelines']) > 0:
    print("\n✓ 2. Generando gráfico histórico coherente...")
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.axis('equal')
    ax.set_xlabel('Eastings (Metros) [EPSG:32618]', fontsize=12)
    ax.set_ylabel('Northings (Metros) [EPSG:32618]', fontsize=12)
    ax.set_title(f'Evolución de la Línea de Costa - {sitename}', fontsize=14)
    ax.grid(linestyle='--', color='gray', alpha=0.7)

    colores = plt.cm.jet(np.linspace(0, 1, len(output['shorelines'])))
    fechas_usadas = []

    for i in range(len(output['shorelines'])):
        sl = output['shorelines'][i]
        date = output['dates'][i]
        if len(sl) > 0:
            fecha_str = date.strftime('%d-%m-%Y')
            etiqueta = fecha_str if fecha_str not in fechas_usadas else None
            if etiqueta: fechas_usadas.append(fecha_str)
            ax.plot(sl[:, 0], sl[:, 1], '.', color=colores[i], markersize=4, label=etiqueta)

    if len(fechas_usadas) > 0:
        ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), title="Fechas", fontsize=8, markerscale=3)
        plt.tight_layout()
        ruta_grafico = os.path.join(filepath, sitename, f'{sitename}_Mapeo.png')
        plt.savefig(ruta_grafico, dpi=300, bbox_inches='tight')
        plt.close(fig)
        display(Image(filename=ruta_grafico))

# C. VIDEO TIMELAPSE
print("\n✓ 3. Creando Video Timelapse Clasificado (.mp4)...")
ruta_video = os.path.join(filepath, sitename, f"{sitename}_timelapse_clasificado.mp4")
if os.path.exists(carpeta_deteccion):
    images_list = sorted(glob.glob(os.path.join(carpeta_deteccion, '*.jpg')))
    if len(images_list) > 0:
        try:
            frames = [imageio.imread(img_path) for img_path in images_list]
            imageio.mimsave(ruta_video, frames, fps=3)
            print(f"¡Video creado con éxito!")
        except Exception as e:
            print(f"Error creando el video: {e}")


# --- INFORME ROBUSTO NATIVO DE COASTSAT ---
print("\n--- GENERANDO INFORME ANALÍTICO (TRANSECTOS NATIVOS) ---")

if len(output['shorelines']) > 0:
    # Seleccionar la primera línea de costa válida para trazar transectos
    idx_validos = [i for i, sl in enumerate(output['shorelines']) if len(sl) > 50]

    if len(idx_validos) > 0:
        L0 = LineString(output['shorelines'][idx_validos[0]])
        transects = dict([])
        espaciado_m = 100       # Un transecto cada 100m
        longitud_mitad = 300    # Largo del transecto (300m al mar, 300m a tierra)

        # Generación automática de transectos
        for i, dist in enumerate(np.arange(10, L0.length, espaciado_m)):
            p = L0.interpolate(dist)
            p_antes = L0.interpolate(max(0, dist - 5))
            p_despues = L0.interpolate(min(L0.length, dist + 5))
            t = np.array([p_despues.x - p_antes.x, p_despues.y - p_antes.y])
            t = t / np.linalg.norm(t) if np.linalg.norm(t) > 0 else np.array([1,0])
            n = np.array([-t[1], t[0]])
            p1 = np.array([p.x, p.y]) - n * longitud_mitad
            p2 = np.array([p.x, p.y]) + n * longitud_mitad
            transects[f'Transecto_{i+1}'] = np.array([p1, p2])

        # Control de Calidad Nativo (QC)
        settings_transects = {
            'along_dist': 25,
            'min_points': 3,
            'max_std': 15,
            'max_range': 30,
            'min_chainage': -100,
            'multiple_inter': 'auto',
            'auto_prc': 0.1,
        }

        # Calcular intersecciones con CoastSat
        cross_distance = SDS_transects.compute_intersection_QC(output, transects, settings_transects)

        # Exportar a CSV tabular
        out_dict = {'dates': output['dates']}
        for key in transects.keys():
            out_dict[key] = cross_distance[key]

        df = pd.DataFrame(out_dict)
        ruta_csv = os.path.join(filepath, sitename, f'{sitename}_reporte_robusto.csv')
        df.to_csv(ruta_csv, sep=',', index=False)
        print(f"✓ Informe CSV creado en: {ruta_csv}")

        # Exportar a GeoJSON
        gdf = SDS_tools.transects_to_gdf(transects)
        gdf.set_crs(epsg=settings['output_epsg'], inplace=True, allow_override=True)
        ruta_geojson = os.path.join(filepath, sitename, f'{sitename}_transectos.geojson')
        gdf.to_file(ruta_geojson, driver='GeoJSON', encoding='utf-8')
        print(f"✓ Archivo GeoJSON de transectos creado en: {ruta_geojson}")

print("\n¡ANÁLISIS COMPLETADO AL 100%!")



---

**4**


---



In [ ]:
import os
import glob
import shutil
import numpy as np
import pandas as pd
import imageio.v2 as imageio
import matplotlib
import matplotlib.pyplot as plt
matplotlib.use('Agg')
from IPython.display import Image, display
from shapely.geometry import LineString
from google.colab import files

# Módulos oficiales de CoastSat
from coastsat import SDS_download, SDS_preprocess, SDS_shoreline, SDS_transects, SDS_tools

# 1. NOMBRE DEL SEGMENTO Y CARPETA
sitename = 'segmento_playas_salgar_principal'
filepath = os.path.join(os.getcwd(), 'data')
carpeta = os.path.join(filepath, sitename)

if os.path.exists(carpeta):
    shutil.rmtree(carpeta)

# 2. COORDENADAS
polygon = [
    [[-74.9534499465,11.004813099],[-74.9568205616,11.0093106509],[-74.9444096268,11.0182725467],[-74.9410390117,11.0137751317],[-74.9534499465,11.004813099]]
]

dates = ['2026-04-29', '2026-06-03']
sat_list = ['S2', 'L8']
inputs = {'polygon': polygon, 'dates': dates, 'sat_list': sat_list, 'sitename': sitename, 'filepath': filepath}

# 3. DESCARGA
print(f"1. Descargando imágenes para el segmento: {sitename}...")
metadata = SDS_download.retrieve_images(inputs)

# 4. CONFIGURACIÓN
settings = {
    'cloud_thresh': 0.1,
    'dist_clouds': 300,
    'output_epsg': 32618,
    'check_detection': False,
    'adjust_detection': False,
    'save_figure': True,
    's2cloudless_prob': 60,
    'sand_color': 'default',
    'cloud_mask_issue': False,
    'pan_off': False,
    'min_beach_area': 1000,
    'min_length_sl': 800,  # Bajado a 800m para asegurar que encaje en tu recuadro
    'inputs': inputs
}

# 5. PREPROCESAMIENTO Y EXTRACCIÓN
print("2. Preprocesando imágenes y filtrando ruido...")
SDS_preprocess.save_jpg(metadata, settings, use_matplotlib=True)

def dummy_maximize(): pass
class DummyWindow:
    def showMaximized(self): pass
class DummyManager: window = DummyWindow()
plt.get_current_fig_manager = lambda: DummyManager()

print("3. Ejecutando extracción de línea de costa...")
output = SDS_shoreline.extract_shorelines(metadata, settings)


# --- GENERACIÓN DE RESULTADOS VISUALES ---
print("\n--- GENERANDO RESULTADOS VISUALES ---")

# A. IMAGEN CLASIFICADA
carpeta_deteccion = os.path.join(filepath, sitename, 'jpg_files', 'detection')
if os.path.exists(carpeta_deteccion):
    imagenes_color = sorted(glob.glob(os.path.join(carpeta_deteccion, '*.jpg')))
    if len(imagenes_color) > 0:
        ultima_img = imagenes_color[-1]
        print("\n✓ 1. Imagen clasificada lista:")
        display(Image(filename=ultima_img))

# B. GRÁFICO HISTÓRICO (MAPEO COHERENTE)
if len(output['shorelines']) > 0:
    print("\n✓ 2. Generando gráfico histórico coherente...")
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.axis('equal')
    ax.set_xlabel('Eastings (Metros) [EPSG:32618]', fontsize=12)
    ax.set_ylabel('Northings (Metros) [EPSG:32618]', fontsize=12)
    ax.set_title(f'Evolución de la Línea de Costa - {sitename}', fontsize=14)
    ax.grid(linestyle='--', color='gray', alpha=0.7)

    colores = plt.cm.jet(np.linspace(0, 1, len(output['shorelines'])))
    fechas_usadas = []

    for i in range(len(output['shorelines'])):
        sl = output['shorelines'][i]
        date = output['dates'][i]
        if len(sl) > 0:
            fecha_str = date.strftime('%d-%m-%Y')
            etiqueta = fecha_str if fecha_str not in fechas_usadas else None
            if etiqueta: fechas_usadas.append(fecha_str)
            ax.plot(sl[:, 0], sl[:, 1], '.', color=colores[i], markersize=4, label=etiqueta)

    if len(fechas_usadas) > 0:
        ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), title="Fechas", fontsize=8, markerscale=3)
        plt.tight_layout()
        ruta_grafico = os.path.join(filepath, sitename, f'{sitename}_Mapeo.png')
        plt.savefig(ruta_grafico, dpi=300, bbox_inches='tight')
        plt.close(fig)
        display(Image(filename=ruta_grafico))

# C. VIDEO TIMELAPSE
print("\n✓ 3. Creando Video Timelapse Clasificado (.mp4)...")
ruta_video = os.path.join(filepath, sitename, f"{sitename}_timelapse_clasificado.mp4")
if os.path.exists(carpeta_deteccion):
    images_list = sorted(glob.glob(os.path.join(carpeta_deteccion, '*.jpg')))
    if len(images_list) > 0:
        try:
            frames = [imageio.imread(img_path) for img_path in images_list]
            imageio.mimsave(ruta_video, frames, fps=3)
            print(f"¡Video creado con éxito!")
        except Exception as e:
            print(f"Error creando el video: {e}")


# --- INFORME ROBUSTO NATIVO DE COASTSAT ---
print("\n--- GENERANDO INFORME ANALÍTICO (TRANSECTOS NATIVOS) ---")

if len(output['shorelines']) > 0:
    # Seleccionar la primera línea de costa válida para trazar transectos
    idx_validos = [i for i, sl in enumerate(output['shorelines']) if len(sl) > 50]

    if len(idx_validos) > 0:
        L0 = LineString(output['shorelines'][idx_validos[0]])
        transects = dict([])
        espaciado_m = 100       # Un transecto cada 100m
        longitud_mitad = 300    # Largo del transecto (300m al mar, 300m a tierra)

        # Generación automática de transectos
        for i, dist in enumerate(np.arange(10, L0.length, espaciado_m)):
            p = L0.interpolate(dist)
            p_antes = L0.interpolate(max(0, dist - 5))
            p_despues = L0.interpolate(min(L0.length, dist + 5))
            t = np.array([p_despues.x - p_antes.x, p_despues.y - p_antes.y])
            t = t / np.linalg.norm(t) if np.linalg.norm(t) > 0 else np.array([1,0])
            n = np.array([-t[1], t[0]])
            p1 = np.array([p.x, p.y]) - n * longitud_mitad
            p2 = np.array([p.x, p.y]) + n * longitud_mitad
            transects[f'Transecto_{i+1}'] = np.array([p1, p2])

        # Control de Calidad Nativo (QC)
        settings_transects = {
            'along_dist': 25,
            'min_points': 3,
            'max_std': 15,
            'max_range': 30,
            'min_chainage': -100,
            'multiple_inter': 'auto',
            'auto_prc': 0.1,
        }

        # Calcular intersecciones con CoastSat
        cross_distance = SDS_transects.compute_intersection_QC(output, transects, settings_transects)

        # Exportar a CSV tabular
        out_dict = {'dates': output['dates']}
        for key in transects.keys():
            out_dict[key] = cross_distance[key]

        df = pd.DataFrame(out_dict)
        ruta_csv = os.path.join(filepath, sitename, f'{sitename}_reporte_robusto.csv')
        df.to_csv(ruta_csv, sep=',', index=False)
        print(f"✓ Informe CSV creado en: {ruta_csv}")

        # Exportar a GeoJSON
        gdf = SDS_tools.transects_to_gdf(transects)
        gdf.set_crs(epsg=settings['output_epsg'], inplace=True, allow_override=True)
        ruta_geojson = os.path.join(filepath, sitename, f'{sitename}_transectos.geojson')
        gdf.to_file(ruta_geojson, driver='GeoJSON', encoding='utf-8')
        print(f"✓ Archivo GeoJSON de transectos creado en: {ruta_geojson}")

print("\n¡ANÁLISIS COMPLETADO AL 100%!")



---
**5**

---




In [ ]:
import os
import glob
import shutil
import numpy as np
import pandas as pd
import imageio.v2 as imageio
import matplotlib
import matplotlib.pyplot as plt
matplotlib.use('Agg')
from IPython.display import Image, display
from shapely.geometry import LineString
from google.colab import files

# Módulos oficiales de CoastSat
from coastsat import SDS_download, SDS_preprocess, SDS_shoreline, SDS_transects, SDS_tools

# 1. NOMBRE DEL SEGMENTO Y CARPETA
sitename = 'segmento_transicion_salgar_sabanilla'
filepath = os.path.join(os.getcwd(), 'data')
carpeta = os.path.join(filepath, sitename)

if os.path.exists(carpeta):
    shutil.rmtree(carpeta)

# 2. COORDENADAS
polygon = [
    [[-74.9416760258,11.0146330008],[-74.9460936435,11.0229934894],[-74.9225721791,11.0349676056],[-74.9181545615,11.0266074574],[-74.9416760258,11.0146330008]]
]

dates = ['2026-04-29', '2026-06-03']
sat_list = ['S2', 'L8']
inputs = {'polygon': polygon, 'dates': dates, 'sat_list': sat_list, 'sitename': sitename, 'filepath': filepath}

# 3. DESCARGA
print(f"1. Descargando imágenes para el segmento: {sitename}...")
metadata = SDS_download.retrieve_images(inputs)

# 4. CONFIGURACIÓN
settings = {
    'cloud_thresh': 0.1,
    'dist_clouds': 300,
    'output_epsg': 32618,
    'check_detection': False,
    'adjust_detection': False,
    'save_figure': True,
    's2cloudless_prob': 60,
    'sand_color': 'default',
    'cloud_mask_issue': False,
    'pan_off': False,
    'min_beach_area': 1000,
    'min_length_sl': 800,  # Bajado a 800m para asegurar que encaje en tu recuadro
    'inputs': inputs
}

# 5. PREPROCESAMIENTO Y EXTRACCIÓN
print("2. Preprocesando imágenes y filtrando ruido...")
SDS_preprocess.save_jpg(metadata, settings, use_matplotlib=True)

def dummy_maximize(): pass
class DummyWindow:
    def showMaximized(self): pass
class DummyManager: window = DummyWindow()
plt.get_current_fig_manager = lambda: DummyManager()

print("3. Ejecutando extracción de línea de costa...")
output = SDS_shoreline.extract_shorelines(metadata, settings)


# --- GENERACIÓN DE RESULTADOS VISUALES ---
print("\n--- GENERANDO RESULTADOS VISUALES ---")

# A. IMAGEN CLASIFICADA
carpeta_deteccion = os.path.join(filepath, sitename, 'jpg_files', 'detection')
if os.path.exists(carpeta_deteccion):
    imagenes_color = sorted(glob.glob(os.path.join(carpeta_deteccion, '*.jpg')))
    if len(imagenes_color) > 0:
        ultima_img = imagenes_color[-1]
        print("\n✓ 1. Imagen clasificada lista:")
        display(Image(filename=ultima_img))

# B. GRÁFICO HISTÓRICO (MAPEO COHERENTE)
if len(output['shorelines']) > 0:
    print("\n✓ 2. Generando gráfico histórico coherente...")
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.axis('equal')
    ax.set_xlabel('Eastings (Metros) [EPSG:32618]', fontsize=12)
    ax.set_ylabel('Northings (Metros) [EPSG:32618]', fontsize=12)
    ax.set_title(f'Evolución de la Línea de Costa - {sitename}', fontsize=14)
    ax.grid(linestyle='--', color='gray', alpha=0.7)

    colores = plt.cm.jet(np.linspace(0, 1, len(output['shorelines'])))
    fechas_usadas = []

    for i in range(len(output['shorelines'])):
        sl = output['shorelines'][i]
        date = output['dates'][i]
        if len(sl) > 0:
            fecha_str = date.strftime('%d-%m-%Y')
            etiqueta = fecha_str if fecha_str not in fechas_usadas else None
            if etiqueta: fechas_usadas.append(fecha_str)
            ax.plot(sl[:, 0], sl[:, 1], '.', color=colores[i], markersize=4, label=etiqueta)

    if len(fechas_usadas) > 0:
        ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), title="Fechas", fontsize=8, markerscale=3)
        plt.tight_layout()
        ruta_grafico = os.path.join(filepath, sitename, f'{sitename}_Mapeo.png')
        plt.savefig(ruta_grafico, dpi=300, bbox_inches='tight')
        plt.close(fig)
        display(Image(filename=ruta_grafico))

# C. VIDEO TIMELAPSE
print("\n✓ 3. Creando Video Timelapse Clasificado (.mp4)...")
ruta_video = os.path.join(filepath, sitename, f"{sitename}_timelapse_clasificado.mp4")
if os.path.exists(carpeta_deteccion):
    images_list = sorted(glob.glob(os.path.join(carpeta_deteccion, '*.jpg')))
    if len(images_list) > 0:
        try:
            frames = [imageio.imread(img_path) for img_path in images_list]
            imageio.mimsave(ruta_video, frames, fps=3)
            print(f"¡Video creado con éxito!")
        except Exception as e:
            print(f"Error creando el video: {e}")


# --- INFORME ROBUSTO NATIVO DE COASTSAT ---
print("\n--- GENERANDO INFORME ANALÍTICO (TRANSECTOS NATIVOS) ---")

if len(output['shorelines']) > 0:
    # Seleccionar la primera línea de costa válida para trazar transectos
    idx_validos = [i for i, sl in enumerate(output['shorelines']) if len(sl) > 50]

    if len(idx_validos) > 0:
        L0 = LineString(output['shorelines'][idx_validos[0]])
        transects = dict([])
        espaciado_m = 100       # Un transecto cada 100m
        longitud_mitad = 300    # Largo del transecto (300m al mar, 300m a tierra)

        # Generación automática de transectos
        for i, dist in enumerate(np.arange(10, L0.length, espaciado_m)):
            p = L0.interpolate(dist)
            p_antes = L0.interpolate(max(0, dist - 5))
            p_despues = L0.interpolate(min(L0.length, dist + 5))
            t = np.array([p_despues.x - p_antes.x, p_despues.y - p_antes.y])
            t = t / np.linalg.norm(t) if np.linalg.norm(t) > 0 else np.array([1,0])
            n = np.array([-t[1], t[0]])
            p1 = np.array([p.x, p.y]) - n * longitud_mitad
            p2 = np.array([p.x, p.y]) + n * longitud_mitad
            transects[f'Transecto_{i+1}'] = np.array([p1, p2])

        # Control de Calidad Nativo (QC)
        settings_transects = {
            'along_dist': 25,
            'min_points': 3,
            'max_std': 15,
            'max_range': 30,
            'min_chainage': -100,
            'multiple_inter': 'auto',
            'auto_prc': 0.1,
        }

        # Calcular intersecciones con CoastSat
        cross_distance = SDS_transects.compute_intersection_QC(output, transects, settings_transects)

        # Exportar a CSV tabular
        out_dict = {'dates': output['dates']}
        for key in transects.keys():
            out_dict[key] = cross_distance[key]

        df = pd.DataFrame(out_dict)
        ruta_csv = os.path.join(filepath, sitename, f'{sitename}_reporte_robusto.csv')
        df.to_csv(ruta_csv, sep=',', index=False)
        print(f"✓ Informe CSV creado en: {ruta_csv}")

        # Exportar a GeoJSON
        gdf = SDS_tools.transects_to_gdf(transects)
        gdf.set_crs(epsg=settings['output_epsg'], inplace=True, allow_override=True)
        ruta_geojson = os.path.join(filepath, sitename, f'{sitename}_transectos.geojson')
        gdf.to_file(ruta_geojson, driver='GeoJSON', encoding='utf-8')
        print(f"✓ Archivo GeoJSON de transectos creado en: {ruta_geojson}")

print("\n¡ANÁLISIS COMPLETADO AL 100%!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')



---

**6**


---



In [ ]:
import os
import glob
import shutil
import numpy as np
import pandas as pd
import imageio.v2 as imageio
import matplotlib
import matplotlib.pyplot as plt
matplotlib.use('Agg')
from IPython.display import Image, display
from shapely.geometry import LineString
from google.colab import files

# Módulos oficiales de CoastSat
from coastsat import SDS_download, SDS_preprocess, SDS_shoreline, SDS_transects, SDS_tools

# 1. NOMBRE DEL SEGMENTO Y CARPETA
sitename = 'segmento_sabanilla_limite_norte'
filepath = os.path.join(os.getcwd(), 'data')
carpeta = os.path.join(filepath, sitename)

if os.path.exists(carpeta):
    shutil.rmtree(carpeta)

# 2. COORDENADAS
polygon = [
    [[-74.9245480735,11.0280959147],[-74.9265597672,11.0457210654],[-74.9216215149,11.0462640319],[-74.9196098211,11.0286389138],[-74.9245480735,11.0280959147]]
]

dates = ['2026-04-29', '2026-06-03']
sat_list = ['S2', 'L8']
inputs = {'polygon': polygon, 'dates': dates, 'sat_list': sat_list, 'sitename': sitename, 'filepath': filepath}

# 3. DESCARGA
print(f"1. Descargando imágenes para el segmento: {sitename}...")
metadata = SDS_download.retrieve_images(inputs)

# 4. CONFIGURACIÓN
settings = {
    'cloud_thresh': 0.1,
    'dist_clouds': 300,
    'output_epsg': 32618,
    'check_detection': False,
    'adjust_detection': False,
    'save_figure': True,
    's2cloudless_prob': 60,
    'sand_color': 'default',
    'cloud_mask_issue': False,
    'pan_off': False,
    'min_beach_area': 1000,
    'min_length_sl': 800,  # Bajado a 800m para asegurar que encaje en tu recuadro
    'inputs': inputs
}

# 5. PREPROCESAMIENTO Y EXTRACCIÓN
print("2. Preprocesando imágenes y filtrando ruido...")
SDS_preprocess.save_jpg(metadata, settings, use_matplotlib=True)

def dummy_maximize(): pass
class DummyWindow:
    def showMaximized(self): pass
class DummyManager: window = DummyWindow()
plt.get_current_fig_manager = lambda: DummyManager()

print("3. Ejecutando extracción de línea de costa...")
output = SDS_shoreline.extract_shorelines(metadata, settings)


# --- GENERACIÓN DE RESULTADOS VISUALES ---
print("\n--- GENERANDO RESULTADOS VISUALES ---")

# A. IMAGEN CLASIFICADA
carpeta_deteccion = os.path.join(filepath, sitename, 'jpg_files', 'detection')
if os.path.exists(carpeta_deteccion):
    imagenes_color = sorted(glob.glob(os.path.join(carpeta_deteccion, '*.jpg')))
    if len(imagenes_color) > 0:
        ultima_img = imagenes_color[-1]
        print("\n✓ 1. Imagen clasificada lista:")
        display(Image(filename=ultima_img))

# B. GRÁFICO HISTÓRICO (MAPEO COHERENTE)
if len(output['shorelines']) > 0:
    print("\n✓ 2. Generando gráfico histórico coherente...")
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.axis('equal')
    ax.set_xlabel('Eastings (Metros) [EPSG:32618]', fontsize=12)
    ax.set_ylabel('Northings (Metros) [EPSG:32618]', fontsize=12)
    ax.set_title(f'Evolución de la Línea de Costa - {sitename}', fontsize=14)
    ax.grid(linestyle='--', color='gray', alpha=0.7)

    colores = plt.cm.jet(np.linspace(0, 1, len(output['shorelines'])))
    fechas_usadas = []

    for i in range(len(output['shorelines'])):
        sl = output['shorelines'][i]
        date = output['dates'][i]
        if len(sl) > 0:
            fecha_str = date.strftime('%d-%m-%Y')
            etiqueta = fecha_str if fecha_str not in fechas_usadas else None
            if etiqueta: fechas_usadas.append(fecha_str)
            ax.plot(sl[:, 0], sl[:, 1], '.', color=colores[i], markersize=4, label=etiqueta)

    if len(fechas_usadas) > 0:
        ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), title="Fechas", fontsize=8, markerscale=3)
        plt.tight_layout()
        ruta_grafico = os.path.join(filepath, sitename, f'{sitename}_Mapeo.png')
        plt.savefig(ruta_grafico, dpi=300, bbox_inches='tight')
        plt.close(fig)
        display(Image(filename=ruta_grafico))

# C. VIDEO TIMELAPSE
print("\n✓ 3. Creando Video Timelapse Clasificado (.mp4)...")
ruta_video = os.path.join(filepath, sitename, f"{sitename}_timelapse_clasificado.mp4")
if os.path.exists(carpeta_deteccion):
    images_list = sorted(glob.glob(os.path.join(carpeta_deteccion, '*.jpg')))
    if len(images_list) > 0:
        try:
            frames = [imageio.imread(img_path) for img_path in images_list]
            imageio.mimsave(ruta_video, frames, fps=3)
            print(f"¡Video creado con éxito!")
        except Exception as e:
            print(f"Error creando el video: {e}")


# --- INFORME ROBUSTO NATIVO DE COASTSAT ---
print("\n--- GENERANDO INFORME ANALÍTICO (TRANSECTOS NATIVOS) ---")

if len(output['shorelines']) > 0:
    # Seleccionar la primera línea de costa válida para trazar transectos
    idx_validos = [i for i, sl in enumerate(output['shorelines']) if len(sl) > 50]

    if len(idx_validos) > 0:
        L0 = LineString(output['shorelines'][idx_validos[0]])
        transects = dict([])
        espaciado_m = 100       # Un transecto cada 100m
        longitud_mitad = 300    # Largo del transecto (300m al mar, 300m a tierra)

        # Generación automática de transectos
        for i, dist in enumerate(np.arange(10, L0.length, espaciado_m)):
            p = L0.interpolate(dist)
            p_antes = L0.interpolate(max(0, dist - 5))
            p_despues = L0.interpolate(min(L0.length, dist + 5))
            t = np.array([p_despues.x - p_antes.x, p_despues.y - p_antes.y])
            t = t / np.linalg.norm(t) if np.linalg.norm(t) > 0 else np.array([1,0])
            n = np.array([-t[1], t[0]])
            p1 = np.array([p.x, p.y]) - n * longitud_mitad
            p2 = np.array([p.x, p.y]) + n * longitud_mitad
            transects[f'Transecto_{i+1}'] = np.array([p1, p2])

        # Control de Calidad Nativo (QC)
        settings_transects = {
            'along_dist': 25,
            'min_points': 3,
            'max_std': 15,
            'max_range': 30,
            'min_chainage': -100,
            'multiple_inter': 'auto',
            'auto_prc': 0.1,
        }

        # Calcular intersecciones con CoastSat
        cross_distance = SDS_transects.compute_intersection_QC(output, transects, settings_transects)

        # Exportar a CSV tabular
        out_dict = {'dates': output['dates']}
        for key in transects.keys():
            out_dict[key] = cross_distance[key]

        df = pd.DataFrame(out_dict)
        ruta_csv = os.path.join(filepath, sitename, f'{sitename}_reporte_robusto.csv')
        df.to_csv(ruta_csv, sep=',', index=False)
        print(f"✓ Informe CSV creado en: {ruta_csv}")

        # Exportar a GeoJSON
        gdf = SDS_tools.transects_to_gdf(transects)
        gdf.set_crs(epsg=settings['output_epsg'], inplace=True, allow_override=True)
        ruta_geojson = os.path.join(filepath, sitename, f'{sitename}_transectos.geojson')
        gdf.to_file(ruta_geojson, driver='GeoJSON', encoding='utf-8')
        print(f"✓ Archivo GeoJSON de transectos creado en: {ruta_geojson}")

print("\n¡ANÁLISIS COMPLETADO AL 100%!")

**FIN**

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ANÁLISIS COMPLETO DESDE PKL — COSTAS_GITT
# ──────────────────────────────────────────────────────────────────────────────
# Genera por cada segmento:
#   1. Mapa histórico de todas las shorelines (coloreado por fecha)
#   2. Video timelapse de las imágenes de detección
#   3. Mapa de vectores de desplazamiento neto (inicio → fin)
#   4. Serie temporal con rolling median centrada + banda ±1σ
#   5. CSV de transectos (reporte_robusto)
#
# NO descarga imágenes ni corre extracción. Solo lee PKLs de Drive.
# ══════════════════════════════════════════════════════════════════════════════

import os
import glob
import pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
import imageio.v2 as imageio
from IPython.display import Image, display
from shapely.geometry import LineString
from coastsat import SDS_tools, SDS_transects
from google.colab import drive
import warnings
warnings.filterwarnings('ignore')

drive.mount('/content/drive')

# ══════════════════════════════════════════════════════════════════════════════
# ▶▶  CONFIG — ajusta aquí  ◀◀
# ══════════════════════════════════════════════════════════════════════════════
filepath    = '/content/drive/MyDrive/coast'
CARPETA_PKL = 'Analisis Abril'   # subcarpeta donde están los PKLs

nombres_segmentos = [
    'segmento_costa_oeste_puerto',
    'segmento_malecon_general',
    'segmento_playas_salgar_principal',
    'segmento_puerto_salgar_amplio',
    'segmento_sabanilla_limite_norte',
    'segmento_transicion_salgar_sabanilla',
]

# ── Parámetros de transectos ──────────────────────────────────────────────────
espaciado_m    = 300
longitud_mitad = 300
min_len_sl     = 80

settings_transects = {
    'along_dist':     25,
    'min_points':      3,
    'max_std':        15,
    'max_range':      30,
    'min_chainage':  -100,
    'multiple_inter': 'auto',
    'auto_prc':       0.1,
}
settings_outliers = {
    'otsu_threshold':   [-0.5, 0],
    'max_cross_change':  20,
    'plot_fig':         False,
}

UMBRAL_ETIQUETA_M = 5
UMBRAL_DIBUJO_M   = 200
ROLLING_DIAS      = 60
PCTL_LO, PCTL_HI  = 10, 90

# ══════════════════════════════════════════════════════════════════════════════
# HELPERS
# ══════════════════════════════════════════════════════════════════════════════
def shoreline_valida(sl, max_jump_factor=10):
    if sl is None or len(sl) < min_len_sl:
        return False
    dif  = np.diff(sl, axis=0)
    step = np.sqrt(dif[:, 0]**2 + dif[:, 1]**2)
    if len(step) == 0:
        return False
    med = np.median(step)
    if med <= 0:
        return False
    return np.max(step) <= max_jump_factor * med


def elegir_extremos(output, idx_validos, frac=0.15):
    """Elige inicio y fin por menor ruido geométrico, no por posición temporal."""
    def ruido(sl):
        dif  = np.diff(sl, axis=0)
        step = np.sqrt(dif[:, 0]**2 + dif[:, 1]**2)
        return float(np.std(step)) if len(step) > 0 else 9e9
    n     = len(idx_validos)
    corte = max(1, int(n * frac))
    idx_ini = min(idx_validos[:corte],  key=lambda i: ruido(output['shorelines'][i]))
    idx_fin = min(idx_validos[-corte:], key=lambda i: ruido(output['shorelines'][i]))
    return idx_ini, idx_fin


def promedio_robusto(df, cols):
    """Media inter-transecto con recorte P10–P90 por fecha."""
    X  = df[cols].copy()
    lo = X.quantile(PCTL_LO / 100.0, axis=1)
    hi = X.quantile(PCTL_HI / 100.0, axis=1)
    for c in cols:
        X[c] = X[c].where((X[c] >= lo) & (X[c] <= hi))
    return X.mean(axis=1, skipna=True)


# ══════════════════════════════════════════════════════════════════════════════
# BUCLE PRINCIPAL
# ══════════════════════════════════════════════════════════════════════════════
for sitename in nombres_segmentos:
    print(f"\n{'='*65}")
    print(f"  {sitename}")
    print(f"{'='*65}")

    carpeta_seg = os.path.join(filepath, sitename)

    # ── Localizar PKL ─────────────────────────────────────────────────────
    carpeta_pkl_dir = os.path.join(carpeta_seg, CARPETA_PKL)
    candidatos = []
    if os.path.isdir(carpeta_pkl_dir):
        candidatos += sorted(glob.glob(
            os.path.join(carpeta_pkl_dir, f'{sitename}*_output*.pkl')
        ), key=os.path.getsize, reverse=True)   # el más grande primero
    candidatos += [
        os.path.join(carpeta_seg, f'{sitename}_output_merged.pkl'),
        os.path.join(carpeta_seg, f'{sitename}_output.pkl'),
    ]

    pkl_path = next((p for p in candidatos if os.path.exists(p)), None)
    if not pkl_path:
        print(f"  ❌  PKL no encontrado.")
        continue

    carpeta_salida = os.path.dirname(pkl_path)
    print(f"  PKL → {os.path.relpath(pkl_path, carpeta_seg)}")

    with open(pkl_path, 'rb') as f:
        output = pickle.load(f)

    output = SDS_tools.remove_duplicates(output)
    output = SDS_tools.remove_inaccurate_georef(output, 10)

    if not output.get('shorelines'):
        print(f"  ❌  Sin shorelines.")
        continue

    print(f"  {output['dates'][0].strftime('%Y-%m-%d')} → "
          f"{output['dates'][-1].strftime('%Y-%m-%d')}  "
          f"({len(output['shorelines'])} fechas)")

    # ── Filtrar shorelines válidas ─────────────────────────────────────────
    idx_validos = sorted(
        [i for i, sl in enumerate(output['shorelines']) if shoreline_valida(sl)],
        key=lambda i: output['dates'][i]
    )
    if len(idx_validos) < 2:
        print(f"  ⚠️  Menos de 2 shorelines válidas.")
        continue
    print(f"  {len(idx_validos)} shorelines válidas de {len(output['shorelines'])} totales")

    # ── Extremos por calidad ───────────────────────────────────────────────
    idx_ini, idx_fin = elegir_extremos(output, idx_validos)
    f_ini = output['dates'][idx_ini].strftime('%Y-%m-%d')
    f_fin = output['dates'][idx_fin].strftime('%Y-%m-%d')

    # ══════════════════════════════════════════════════════════════════════
    # GRÁFICO 1 — Mapa histórico de TODAS las shorelines válidas
    # ══════════════════════════════════════════════════════════════════════
    print(f"\n  [1/4] Mapa histórico de shorelines...")

    fig, ax = plt.subplots(figsize=(11, 9))
    ax.axis('equal')
    ax.set_facecolor('#f0f4f8')
    ax.grid(linestyle=':', color='0.7', alpha=0.5)

    colores = plt.cm.RdYlGn(np.linspace(0.05, 0.95, len(idx_validos)))

    for orden, idx in enumerate(idx_validos):
        sl   = output['shorelines'][idx]
        fecha = output['dates'][idx].strftime('%Y-%m')
        ax.plot(sl[:, 0], sl[:, 1], '.', color=colores[orden],
                markersize=2.5, alpha=0.75)

    # Destacar inicio y fin
    sl_i = output['shorelines'][idx_ini]
    sl_f = output['shorelines'][idx_fin]
    ax.plot(sl_i[:, 0], sl_i[:, 1], '-', color='steelblue',
            lw=2, label=f'Inicio seleccionado ({f_ini})', zorder=5)
    ax.plot(sl_f[:, 0], sl_f[:, 1], '-', color='firebrick',
            lw=2, label=f'Fin seleccionado ({f_fin})',   zorder=5)

    # Barra de color como leyenda temporal
    sm = plt.cm.ScalarMappable(
        cmap='RdYlGn',
        norm=plt.Normalize(
            vmin=output['dates'][idx_validos[0]].year,
            vmax=output['dates'][idx_validos[-1]].year
        )
    )
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
    cbar.set_label('Año', fontsize=10)

    ax.set_title(
        f'Evolución histórica de la línea de costa — {sitename}\n'
        f'{len(idx_validos)} imágenes válidas · {f_ini} → {f_fin}',
        fontsize=12, fontweight='bold'
    )
    ax.set_xlabel('Eastings [EPSG:32618]', fontsize=10)
    ax.set_ylabel('Northings [EPSG:32618]', fontsize=10)
    ax.legend(loc='lower left', fontsize=9, framealpha=0.9)

    plt.tight_layout()
    ruta_hist = os.path.join(carpeta_salida, f'{sitename}_Historico_Shorelines.png')
    plt.savefig(ruta_hist, dpi=300, bbox_inches='tight')
    plt.close(fig)
    display(Image(filename=ruta_hist))

    # ══════════════════════════════════════════════════════════════════════
    # GRÁFICO 2 — Video timelapse desde jpg_files/detection
    # ══════════════════════════════════════════════════════════════════════
    print(f"\n  [2/4] Video timelapse...")

    carpeta_det = os.path.join(carpeta_seg, 'jpg_files', 'detection')
    if os.path.isdir(carpeta_det):
        imagenes_jpg = sorted(glob.glob(os.path.join(carpeta_det, '*.jpg')))
        if imagenes_jpg:
            try:
                frames = [imageio.imread(p) for p in imagenes_jpg]
                ruta_video = os.path.join(
                    carpeta_salida,
                    f'{sitename}_timelapse.mp4'
                )
                imageio.mimsave(ruta_video, frames, fps=4, quality=7)
                print(f"  ✅  Video → {os.path.relpath(ruta_video, carpeta_seg)}")
                print(f"      {len(frames)} frames · {len(frames)/4:.0f}s a 4fps")
            except Exception as e:
                print(f"  ⚠️  Error en video: {e}")
        else:
            print(f"  ⚠️  Sin JPGs en jpg_files/detection/")
    else:
        print(f"  ⚠️  Carpeta jpg_files/detection/ no existe.")

    # ── Transectos (necesarios para gráficos 3 y 4) ───────────────────────
    idx_base = idx_validos[len(idx_validos) // 2]
    L0 = LineString(output['shorelines'][idx_base])
    if L0.length <= 0:
        print(f"  ⚠️  Shoreline base sin longitud — saltando transectos.")
        continue

    transects = {}
    for i, dist in enumerate(np.arange(10, L0.length, espaciado_m)):
        p       = L0.interpolate(dist)
        p_antes = L0.interpolate(max(0, dist - 5))
        p_desp  = L0.interpolate(min(L0.length, dist + 5))
        t = np.array([p_desp.x - p_antes.x, p_desp.y - p_antes.y])
        nrm = np.linalg.norm(t)
        t = t / nrm if nrm > 0 else np.array([1.0, 0.0])
        n = np.array([-t[1], t[0]])
        p1 = np.array([p.x, p.y]) - n * longitud_mitad
        p2 = np.array([p.x, p.y]) + n * longitud_mitad
        transects[f'T_{i+1}'] = np.array([p1, p2])

    cross = SDS_transects.compute_intersection_QC(output, transects, settings_transects)
    if not cross:
        print(f"  ⚠️  Sin datos tras QC.")
        continue
    cross = SDS_transects.reject_outliers(cross, output, settings_outliers)
    if not cross:
        print(f"  ⚠️  Sin datos tras reject_outliers.")
        continue

    df = pd.DataFrame({'dates': output['dates']})
    for k, v in cross.items():
        df[k] = v

    cols_ok = [
        k for k in cross
        if len(df[['dates', k]].dropna()) >= 2
        and abs(float(df[k].dropna().iloc[-1]) - float(df[k].dropna().iloc[0])) <= UMBRAL_DIBUJO_M
    ]
    if not cols_ok:
        print(f"  ⚠️  Todos los transectos superan {UMBRAL_DIBUJO_M} m.")
        continue
    print(f"\n  {len(cols_ok)} transectos válidos")

    # ── CSV de transectos ──────────────────────────────────────────────────
    ruta_csv = os.path.join(carpeta_salida, f'{sitename}_reporte_robusto.csv')
    df[['dates'] + cols_ok].to_csv(ruta_csv, index=False)
    print(f"  CSV → {os.path.relpath(ruta_csv, carpeta_seg)}")

    # ══════════════════════════════════════════════════════════════════════
    # GRÁFICO 3 — Mapa de vectores de desplazamiento neto
    # ══════════════════════════════════════════════════════════════════════
    print(f"\n  [3/4] Mapa de vectores...")

    fig, ax = plt.subplots(figsize=(13, 10))
    ax.axis('equal')
    ax.set_facecolor('#f0f4f8')
    ax.grid(linestyle=':', color='0.7', alpha=0.5)

    sl_ini = output['shorelines'][idx_ini]
    sl_fin = output['shorelines'][idx_fin]

    ax.plot(sl_ini[:, 0], sl_ini[:, 1], '-', color='steelblue',
            lw=2.2, label=f'Costa inicial ({f_ini})', zorder=4)
    ax.plot(sl_fin[:, 0], sl_fin[:, 1], '-', color='firebrick',
            lw=2.2, label=f'Costa final ({f_fin})',   zorder=4)

    n_acrecion = n_erosion = 0

    for col in cols_ok:
        valido = df[['dates', col]].dropna()
        if len(valido) < 2:
            continue
        d0, d1  = float(valido[col].iloc[0]), float(valido[col].iloc[-1])
        cambio  = d1 - d0

        p1, p2  = transects[col][0], transects[col][1]
        vec     = p2 - p1
        nrm     = np.linalg.norm(vec)
        if nrm == 0:
            continue
        u   = vec / nrm
        pt0 = p1 + u * d0
        pt1 = p1 + u * d1

        color_v = '#2d9e2d' if cambio >= 0 else '#c0392b'
        alpha_v = 0.35 + min(abs(cambio) / UMBRAL_DIBUJO_M, 0.65)

        ax.annotate('', xy=(pt1[0], pt1[1]), xytext=(pt0[0], pt0[1]),
                    arrowprops=dict(arrowstyle='->', color=color_v,
                                   lw=1.8, alpha=alpha_v), zorder=3)
        ax.plot(pt0[0], pt0[1], 'o', color=color_v, ms=3.5, alpha=0.7, zorder=3)

        if abs(cambio) >= UMBRAL_ETIQUETA_M:
            mid   = (pt0 + pt1) / 2
            signo = '+' if cambio >= 0 else ''
            ax.text(mid[0], mid[1], f'{signo}{cambio:.1f} m',
                    color=color_v, fontsize=9, fontweight='bold',
                    ha='center', va='center', zorder=5,
                    bbox=dict(facecolor='white', alpha=0.82,
                              edgecolor='none', pad=1.5))
        if cambio >= 0:
            n_acrecion += 1
        else:
            n_erosion += 1

    ax.set_title(
        f'Vectores de desplazamiento neto — {sitename}\n'
        f'Período: {f_ini} → {f_fin}',
        fontsize=13, fontweight='bold', pad=14
    )
    ax.set_xlabel('Eastings [EPSG:32618]', fontsize=10)
    ax.set_ylabel('Northings [EPSG:32618]', fontsize=10)
    g_p = mpatches.Patch(color='#2d9e2d',
                         label=f'Ganancia / Acreción (+m) — {n_acrecion} transectos')
    r_p = mpatches.Patch(color='#c0392b',
                         label=f'Pérdida / Erosión (−m) — {n_erosion} transectos')
    h, l = ax.get_legend_handles_labels()
    ax.legend(handles=h + [g_p, r_p], labels=l + [g_p.get_label(), r_p.get_label()],
              loc='lower left', frameon=True, fontsize=9, framealpha=0.9)

    plt.tight_layout()
    ruta_vec = os.path.join(carpeta_salida, f'{sitename}_Mapa_Vectores_Metros.png')
    plt.savefig(ruta_vec, dpi=300, bbox_inches='tight')
    plt.close(fig)
    display(Image(filename=ruta_vec))

    # ══════════════════════════════════════════════════════════════════════
    # GRÁFICO 4 — Serie temporal (rolling median centrada + banda ±1σ)
    # ══════════════════════════════════════════════════════════════════════
    print(f"\n  [4/4] Serie temporal...")

    tendencia = promedio_robusto(df, cols_ok)
    n_ref     = max(1, int(len(tendencia.dropna()) * 0.10))
    ref       = float(tendencia.dropna().iloc[:n_ref].median())
    tendencia = tendencia - ref

    df_s   = pd.DataFrame({'val': tendencia.values},
                           index=pd.to_datetime(df['dates']))
    df_s   = df_s.sort_index()
    smooth = df_s['val'].rolling(
        f'{ROLLING_DIAS}D', min_periods=3, center=True
    ).median()

    fig2, ax2 = plt.subplots(figsize=(13, 5))
    ax2.set_facecolor('#f0f4f8')
    ax2.grid(linestyle=':', color='0.7', alpha=0.5)

    std_serie = df[cols_ok].std(axis=1, skipna=True)
    ax2.fill_between(
        df_s.index,
        df_s['val'] - std_serie.values,
        df_s['val'] + std_serie.values,
        color='steelblue', alpha=0.12,
        label='±1σ variabilidad entre transectos'
    )
    ax2.fill_between(smooth.index, 0, smooth,
                     where=(smooth >= 0),
                     color='#2d9e2d', alpha=0.18, label='Acreción (suavizado)')
    ax2.fill_between(smooth.index, 0, smooth,
                     where=(smooth < 0),
                     color='#c0392b', alpha=0.18, label='Erosión (suavizado)')
    ax2.plot(df_s.index, df_s['val'],
             '+', color='#888', ms=4, alpha=0.4,
             label='Datos filtrados (por fecha)')
    ax2.plot(smooth.index, smooth,
             '-', color='#4c1d95', lw=2.5,
             label=f'Mediana móvil {ROLLING_DIAS}d (centrada)')
    ax2.axhline(0, color='black', ls='--', lw=1.2, alpha=0.6)

    ax2.xaxis.set_major_locator(mdates.YearLocator())
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax2.xaxis.set_minor_locator(mdates.MonthLocator(bymonth=[4, 7, 10]))
    plt.setp(ax2.xaxis.get_majorticklabels(), rotation=0, ha='center')

    if smooth.dropna().shape[0] >= 2:
        delta_tot = float(smooth.dropna().iloc[-1]) - float(smooth.dropna().iloc[0])
        signo_d   = '+' if delta_tot >= 0 else ''
        color_d   = '#2d9e2d' if delta_tot >= 0 else '#c0392b'
        ax2.text(0.015, 0.97,
                 f'Δ total: {signo_d}{delta_tot:.1f} m  ({f_ini} → {f_fin})',
                 transform=ax2.transAxes, fontsize=9.5,
                 va='top', color=color_d, fontweight='bold',
                 bbox=dict(facecolor='white', alpha=0.85,
                           edgecolor='none', pad=3))

    ax2.set_title(
        f'Evolución de la línea de costa — {sitename}',
        fontsize=13, fontweight='bold'
    )
    ax2.set_ylabel('Desplazamiento [m]\n(+ acreción  /  − erosión)', fontsize=10)
    ax2.legend(loc='lower left', fontsize=8.5, framealpha=0.9)

    plt.tight_layout()
    ruta_tend = os.path.join(carpeta_salida, f'{sitename}_Tendencia_Erosion.png')
    plt.savefig(ruta_tend, dpi=300, bbox_inches='tight')
    plt.close(fig2)
    display(Image(filename=ruta_tend))

    print(f"\n  ✅  {sitename} — completado")
    print(f"      Guardado en: {os.path.relpath(carpeta_salida, filepath)}/")

print(f"\n{'='*65}")
print("  ✅  TODOS LOS SEGMENTOS PROCESADOS")
print(f"{'='*65}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ANÁLISIS COMPLETO — COSTAS_GITT
# Genera por segmento los 5 productos del informe:
#   1. Mapa de vectores de desplazamiento  (_Mapa_Vectores_Metros.png)
#   2. Perfil longitudinal de barras       (_Perfil_Longitudinal_Barras.png)
#   3. Tasa anual LRR                      (_Tasa_Anual_LRR.png)
#   4. Heatmap espacio-temporal Hovmöller  (_Heatmap_EspacioTemporal.png)
#   5. Balance de área                     (_Balance_Area.txt)
# ══════════════════════════════════════════════════════════════════════════════

import os
import glob
import pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
from matplotlib.colors import TwoSlopeNorm
from IPython.display import Image, display
from shapely.geometry import LineString, Polygon
from scipy import stats as scipy_stats
from coastsat import SDS_tools, SDS_transects
from google.colab import drive
import warnings
warnings.filterwarnings('ignore')

drive.mount('/content/drive')

# ══════════════════════════════════════════════════════════════════════════════
# ▶▶  CONFIG  ◀◀
# ══════════════════════════════════════════════════════════════════════════════
filepath    = '/content/drive/MyDrive/coast'
CARPETA_PKL = 'Analisis Abril'

nombres_segmentos = [
    'segmento_costa_oeste_puerto',
    'segmento_malecon_general',
    'segmento_playas_salgar_principal',
    'segmento_puerto_salgar_amplio',
    'segmento_sabanilla_limite_norte',
    'segmento_transicion_salgar_sabanilla',
]

espaciado_m    = 300
longitud_mitad = 300
min_len_sl     = 80

settings_transects = {
    'along_dist': 25, 'min_points': 3, 'max_std': 15, 'max_range': 30,
    'min_chainage': -100, 'multiple_inter': 'auto', 'auto_prc': 0.1,
}
settings_outliers = {
    'otsu_threshold': [-0.5, 0], 'max_cross_change': 20, 'plot_fig': False,
}

UMBRAL_ETIQUETA_M = 5
UMBRAL_DIBUJO_M   = 200
ROLLING_DIAS      = 60
PCTL_LO, PCTL_HI  = 10, 90


# ══════════════════════════════════════════════════════════════════════════════
# HELPERS
# ══════════════════════════════════════════════════════════════════════════════
def shoreline_valida(sl, max_jump_factor=10):
    if sl is None or len(sl) < min_len_sl:
        return False
    dif  = np.diff(sl, axis=0)
    step = np.sqrt(dif[:, 0]**2 + dif[:, 1]**2)
    if len(step) == 0:
        return False
    med = np.median(step)
    return med > 0 and np.max(step) <= max_jump_factor * med


def elegir_extremos(output, idx_validos, frac=0.15):
    def ruido(sl):
        dif  = np.diff(sl, axis=0)
        step = np.sqrt(dif[:, 0]**2 + dif[:, 1]**2)
        return float(np.std(step)) if len(step) > 0 else 9e9
    n     = len(idx_validos)
    corte = max(1, int(n * frac))
    ini   = min(idx_validos[:corte],  key=lambda i: ruido(output['shorelines'][i]))
    fin   = min(idx_validos[-corte:], key=lambda i: ruido(output['shorelines'][i]))
    return ini, fin


def promedio_robusto(df, cols):
    X  = df[cols].copy()
    lo = X.quantile(PCTL_LO / 100.0, axis=1)
    hi = X.quantile(PCTL_HI / 100.0, axis=1)
    for c in cols:
        X[c] = X[c].where((X[c] >= lo) & (X[c] <= hi))
    return X.mean(axis=1, skipna=True)


def lrr(fechas, valores):
    """Linear Regression Rate en m/año para una serie temporal."""
    if len(fechas) < 2:
        return np.nan, np.nan
    t = np.array([(d - fechas[0]).days / 365.25 for d in fechas])
    mask = ~np.isnan(valores)
    if mask.sum() < 2:
        return np.nan, np.nan
    slope, _, _, _, stderr = scipy_stats.linregress(t[mask], valores[mask])
    return float(slope), float(stderr)   # m/año, error estándar


def area_entre_shorelines(sl1, sl2):
    """
    Estimación del área ganada/perdida entre dos shorelines (m²).
    Usa el polígono formado por ambas líneas y calcula su área con Shapely.
    Si las líneas tienen distinto nº de puntos, interpola sl2 a la misma longitud.
    """
    try:
        n = min(len(sl1), len(sl2))
        if n < 3:
            return 0.0, 0.0
        # Remuestrear ambas a n puntos equiespaciados
        def resample(sl, n):
            ls  = LineString(sl)
            pts = [ls.interpolate(ls.length * i / (n - 1)) for i in range(n)]
            return np.array([[p.x, p.y] for p in pts])
        a = resample(sl1, n)
        b = resample(sl2, n)
        # Polígono: recorremos sl1 de inicio a fin y sl2 de fin a inicio
        coords = list(map(tuple, a)) + list(map(tuple, b[::-1]))
        poly   = Polygon(coords)
        if not poly.is_valid:
            poly = poly.buffer(0)
        total  = poly.area
        # Separar ganancia y pérdida por transecto (signo de b-a en dirección normal)
        diffs  = np.linalg.norm(b - a, axis=1)
        signs  = np.sign((b - a)[:, 0] * (-(a[:, 1] - np.mean(a[:, 1])))
                         + (b - a)[:, 1] * (a[:, 0] - np.mean(a[:, 0])))
        gan    = float(np.sum(diffs[signs >= 0]) * (total / max(np.sum(diffs), 1e-6)))
        per    = float(np.sum(diffs[signs <  0]) * (total / max(np.sum(diffs), 1e-6)))
        return gan, per
    except Exception:
        return 0.0, 0.0


# ══════════════════════════════════════════════════════════════════════════════
# BUCLE PRINCIPAL
# ══════════════════════════════════════════════════════════════════════════════
for sitename in nombres_segmentos:
    print(f"\n{'='*65}")
    print(f"  {sitename}")
    print(f"{'='*65}")

    carpeta_seg = os.path.join(filepath, sitename)
    dir_pkl     = os.path.join(carpeta_seg, CARPETA_PKL)

    candidatos = (
        sorted(glob.glob(os.path.join(dir_pkl, f'{sitename}*_output*.pkl')),
               key=os.path.getsize, reverse=True)
        if os.path.isdir(dir_pkl) else []
    ) + [
        os.path.join(carpeta_seg, f'{sitename}_output_merged.pkl'),
        os.path.join(carpeta_seg, f'{sitename}_output.pkl'),
    ]

    pkl_path = next((p for p in candidatos if os.path.exists(p)), None)
    if not pkl_path:
        print(f"  ❌  PKL no encontrado.")
        continue

    carpeta_out = os.path.dirname(pkl_path)
    print(f"  PKL → {os.path.relpath(pkl_path, carpeta_seg)}")

    with open(pkl_path, 'rb') as f:
        output = pickle.load(f)
    output = SDS_tools.remove_duplicates(output)
    output = SDS_tools.remove_inaccurate_georef(output, 10)

    if not output.get('shorelines'):
        print(f"  ❌  Sin shorelines.")
        continue

    idx_validos = sorted(
        [i for i, sl in enumerate(output['shorelines']) if shoreline_valida(sl)],
        key=lambda i: output['dates'][i]
    )
    if len(idx_validos) < 2:
        print(f"  ⚠️  < 2 shorelines válidas.")
        continue

    idx_ini, idx_fin = elegir_extremos(output, idx_validos)
    f_ini = output['dates'][idx_ini].strftime('%Y-%m-%d')
    f_fin = output['dates'][idx_fin].strftime('%Y-%m-%d')
    print(f"  Período: {f_ini} → {f_fin}  ({len(idx_validos)} imágenes válidas)")

    # ── Transectos ────────────────────────────────────────────────────────
    idx_base = idx_validos[len(idx_validos) // 2]
    L0 = LineString(output['shorelines'][idx_base])
    if L0.length <= 0:
        print(f"  ⚠️  Shoreline base sin longitud.")
        continue

    transects = {}
    for i, dist in enumerate(np.arange(10, L0.length, espaciado_m)):
        p      = L0.interpolate(dist)
        p_ant  = L0.interpolate(max(0, dist - 5))
        p_dep  = L0.interpolate(min(L0.length, dist + 5))
        t = np.array([p_dep.x - p_ant.x, p_dep.y - p_ant.y])
        nrm = np.linalg.norm(t)
        t = t / nrm if nrm > 0 else np.array([1.0, 0.0])
        n = np.array([-t[1], t[0]])
        p1 = np.array([p.x, p.y]) - n * longitud_mitad
        p2 = np.array([p.x, p.y]) + n * longitud_mitad
        transects[f'T_{i+1}'] = np.array([p1, p2])

    cross = SDS_transects.compute_intersection_QC(output, transects, settings_transects)
    if not cross:
        print(f"  ⚠️  Sin datos tras QC.")
        continue
    cross = SDS_transects.reject_outliers(cross, output, settings_outliers)
    if not cross:
        print(f"  ⚠️  Sin datos tras reject_outliers.")
        continue

    df = pd.DataFrame({'dates': output['dates']})
    for k, v in cross.items():
        df[k] = v

    cols_ok = [
        k for k in cross
        if len(df[['dates', k]].dropna()) >= 2
        and abs(float(df[k].dropna().iloc[-1]) - float(df[k].dropna().iloc[0])) <= UMBRAL_DIBUJO_M
    ]
    if not cols_ok:
        print(f"  ⚠️  Todos los transectos superan {UMBRAL_DIBUJO_M} m.")
        continue
    print(f"  {len(cols_ok)} transectos válidos")

    # Cambio neto por transecto (para productos 1, 2, 3)
    cambios_netos = {}
    for col in cols_ok:
        valido = df[['dates', col]].dropna()
        if len(valido) < 2:
            continue
        cambios_netos[col] = float(valido[col].iloc[-1]) - float(valido[col].iloc[0])

    # LRR por transecto (para producto 3)
    lrr_vals   = {}
    lrr_errors = {}
    for col in cols_ok:
        valido = df[['dates', col]].dropna()
        if len(valido) < 2:
            continue
        slope, err = lrr(list(valido['dates']), valido[col].values)
        lrr_vals[col]   = slope
        lrr_errors[col] = err

    etiquetas_t = [c.replace('T_', 'T') for c in cols_ok]

    # ══════════════════════════════════════════════════════════════════════
    # PRODUCTO 1 — Mapa de vectores
    # ══════════════════════════════════════════════════════════════════════
    print(f"  [1/5] Mapa de vectores...")
    fig, ax = plt.subplots(figsize=(13, 10))
    ax.axis('equal')
    ax.set_facecolor('#f0f4f8')
    ax.grid(':', color='0.7', alpha=0.5)

    sl_ini = output['shorelines'][idx_ini]
    sl_fin = output['shorelines'][idx_fin]
    ax.plot(sl_ini[:, 0], sl_ini[:, 1], '-', color='steelblue', lw=2.2,
            label=f'Costa inicial ({f_ini})', zorder=4)
    ax.plot(sl_fin[:, 0], sl_fin[:, 1], '-', color='firebrick', lw=2.2,
            label=f'Costa final ({f_fin})', zorder=4)

    n_acr = n_ero = 0
    for col in cols_ok:
        cambio = cambios_netos.get(col)
        if cambio is None:
            continue
        valido = df[['dates', col]].dropna()
        d0, d1 = float(valido[col].iloc[0]), float(valido[col].iloc[-1])
        p1, p2 = transects[col][0], transects[col][1]
        vec = p2 - p1;  nrm = np.linalg.norm(vec)
        if nrm == 0: continue
        u = vec / nrm
        pt0, pt1 = p1 + u * d0, p1 + u * d1
        color_v  = '#2d9e2d' if cambio >= 0 else '#c0392b'
        alpha_v  = 0.35 + min(abs(cambio) / UMBRAL_DIBUJO_M, 0.65)
        ax.annotate('', xy=(pt1[0], pt1[1]), xytext=(pt0[0], pt0[1]),
                    arrowprops=dict(arrowstyle='->', color=color_v,
                                   lw=1.8, alpha=alpha_v), zorder=3)
        ax.plot(pt0[0], pt0[1], 'o', color=color_v, ms=3.5, alpha=0.7, zorder=3)
        if abs(cambio) >= UMBRAL_ETIQUETA_M:
            mid = (pt0 + pt1) / 2
            ax.text(mid[0], mid[1], f'{"+" if cambio>=0 else ""}{cambio:.1f} m',
                    color=color_v, fontsize=9, fontweight='bold',
                    ha='center', va='center', zorder=5,
                    bbox=dict(facecolor='white', alpha=0.82, edgecolor='none', pad=1.5))
        n_acr += cambio >= 0;  n_ero += cambio < 0

    g_p = mpatches.Patch(color='#2d9e2d', label=f'Ganancia (+m) — {n_acr} transectos')
    r_p = mpatches.Patch(color='#c0392b', label=f'Pérdida (−m)  — {n_ero} transectos')
    h, l = ax.get_legend_handles_labels()
    ax.legend(handles=h+[g_p,r_p], labels=l+[g_p.get_label(),r_p.get_label()],
              loc='lower left', fontsize=9, framealpha=0.9)
    ax.set_title(f'Vectores de desplazamiento neto — {sitename}\n'
                 f'Período: {f_ini} → {f_fin}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Eastings [EPSG:32618]', fontsize=10)
    ax.set_ylabel('Northings [EPSG:32618]', fontsize=10)
    plt.tight_layout()
    ruta = os.path.join(carpeta_out, f'{sitename}_Mapa_Vectores_Metros.png')
    plt.savefig(ruta, dpi=300, bbox_inches='tight');  plt.close(fig)
    display(Image(filename=ruta))

    # ══════════════════════════════════════════════════════════════════════
    # PRODUCTO 2 — Perfil longitudinal de barras
    # ══════════════════════════════════════════════════════════════════════
    print(f"  [2/5] Perfil longitudinal...")
    vals   = [cambios_netos.get(c, np.nan) for c in cols_ok]
    colors = ['#2d9e2d' if v >= 0 else '#c0392b' for v in vals]
    x      = np.arange(len(cols_ok))

    fig, ax = plt.subplots(figsize=(max(8, len(cols_ok) * 0.7 + 2), 5))
    ax.set_facecolor('#f0f4f8')
    ax.grid(axis='y', linestyle=':', color='0.7', alpha=0.6)
    bars = ax.bar(x, vals, color=colors, edgecolor='white', linewidth=0.6, width=0.7)

    for bar, val in zip(bars, vals):
        if np.isnan(val): continue
        ypos = val + (0.3 if val >= 0 else -0.3)
        ax.text(bar.get_x() + bar.get_width()/2, ypos,
                f'{"+" if val>=0 else ""}{val:.1f}',
                ha='center', va='bottom' if val >= 0 else 'top',
                fontsize=8, fontweight='bold',
                color='#2d9e2d' if val >= 0 else '#c0392b')

    ax.axhline(0, color='black', lw=1.2, alpha=0.7)
    ax.set_xticks(x);  ax.set_xticklabels(etiquetas_t, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Cambio neto [m]\n(+ acreción / − erosión)', fontsize=10)
    ax.set_title(f'Perfil longitudinal de cambio neto — {sitename}\n'
                 f'{f_ini} → {f_fin}', fontsize=12, fontweight='bold')
    g_p2 = mpatches.Patch(color='#2d9e2d', label='Ganancia / Acreción')
    r_p2 = mpatches.Patch(color='#c0392b', label='Pérdida / Erosión')
    ax.legend(handles=[g_p2, r_p2], fontsize=9, framealpha=0.9)
    plt.tight_layout()
    ruta = os.path.join(carpeta_out, f'{sitename}_Perfil_Longitudinal_Barras.png')
    plt.savefig(ruta, dpi=300, bbox_inches='tight');  plt.close(fig)
    display(Image(filename=ruta))

    # ══════════════════════════════════════════════════════════════════════
    # PRODUCTO 3 — Tasa anual LRR
    # ══════════════════════════════════════════════════════════════════════
    print(f"  [3/5] Tasa anual LRR...")
    lrr_list = [lrr_vals.get(c, np.nan) for c in cols_ok]
    err_list = [lrr_errors.get(c, 0.0)  for c in cols_ok]
    colors_l = ['#2d9e2d' if (v >= 0 and not np.isnan(v)) else '#c0392b' for v in lrr_list]

    fig, ax = plt.subplots(figsize=(max(8, len(cols_ok) * 0.7 + 2), 5))
    ax.set_facecolor('#f0f4f8')
    ax.grid(axis='y', linestyle=':', color='0.7', alpha=0.6)
    ax.bar(x, lrr_list, color=colors_l, edgecolor='white', linewidth=0.6,
           width=0.7, yerr=err_list, capsize=3,
           error_kw=dict(ecolor='#555', elinewidth=1, capthick=1))
    ax.axhline(0, color='black', lw=1.2, alpha=0.7)

    for i, (val, err) in enumerate(zip(lrr_list, err_list)):
        if np.isnan(val): continue
        ypos = val + err + (0.05 if val >= 0 else -0.05)
        ax.text(i, ypos, f'{"+" if val>=0 else ""}{val:.2f}',
                ha='center', va='bottom' if val >= 0 else 'top',
                fontsize=7.5, fontweight='bold',
                color='#2d9e2d' if val >= 0 else '#c0392b')

    ax.set_xticks(x);  ax.set_xticklabels(etiquetas_t, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Tasa de cambio [m/año]\n(LRR — regresión lineal)', fontsize=10)
    ax.set_title(f'Tasa anual de desplazamiento (LRR) — {sitename}\n'
                 f'Barras de error = ±1σ de la regresión', fontsize=12, fontweight='bold')
    g_p3 = mpatches.Patch(color='#2d9e2d', label='Acreción (m/año > 0)')
    r_p3 = mpatches.Patch(color='#c0392b', label='Erosión  (m/año < 0)')
    ax.legend(handles=[g_p3, r_p3], fontsize=9, framealpha=0.9)
    plt.tight_layout()
    ruta = os.path.join(carpeta_out, f'{sitename}_Tasa_Anual_LRR.png')
    plt.savefig(ruta, dpi=300, bbox_inches='tight');  plt.close(fig)
    display(Image(filename=ruta))

    # ══════════════════════════════════════════════════════════════════════
    # PRODUCTO 4 — Heatmap espacio-temporal (Hovmöller)
    # ══════════════════════════════════════════════════════════════════════
    print(f"  [4/5] Heatmap espacio-temporal...")

    # Suavizado rolling median para el heatmap
    df_heat = df[['dates'] + cols_ok].copy()
    df_heat = df_heat.set_index('dates').sort_index()

    # Rellenar con rolling median para visualización continua
    df_smooth = df_heat.apply(
        lambda col: col.rolling(f'{ROLLING_DIAS}D', min_periods=2, center=True).median()
    )

    # Centrar cada transecto en 0 (referencia al inicio)
    for c in cols_ok:
        first = df_smooth[c].dropna()
        if not first.empty:
            df_smooth[c] -= float(first.iloc[0])

    Z = df_smooth[cols_ok].values.T   # shape: (n_transectos, n_fechas)

    # Límite simétrico para la escala de color
    vlim = np.nanpercentile(np.abs(Z), 95)
    vlim = max(vlim, 1.0)

    fig, ax = plt.subplots(figsize=(14, max(4, len(cols_ok) * 0.45 + 2)))
    norm  = TwoSlopeNorm(vmin=-vlim, vcenter=0, vmax=vlim)
    fechas_num = mdates.date2num(df_smooth.index.to_pydatetime())

    im = ax.imshow(
        Z,
        aspect='auto',
        extent=[fechas_num[0], fechas_num[-1], len(cols_ok) - 0.5, -0.5],
        cmap='RdYlGn',
        norm=norm,
        interpolation='nearest'
    )

    ax.xaxis_date()
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_minor_locator(mdates.MonthLocator(bymonth=[4, 7, 10]))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=0, ha='center')

    ax.set_yticks(range(len(cols_ok)))
    ax.set_yticklabels(etiquetas_t, fontsize=8)
    ax.set_ylabel('Transecto', fontsize=10)
    ax.set_title(f'Heatmap espacio-temporal — {sitename}\n'
                 f'Desplazamiento relativo de la línea costera [m]  ·  '
                 f'Mediana móvil {ROLLING_DIAS}d',
                 fontsize=12, fontweight='bold')

    cbar = plt.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
    cbar.set_label('Desplazamiento [m]\n(+ acreción  /  − erosión)', fontsize=9)

    plt.tight_layout()
    ruta = os.path.join(carpeta_out, f'{sitename}_Heatmap_EspacioTemporal.png')
    plt.savefig(ruta, dpi=300, bbox_inches='tight');  plt.close(fig)
    display(Image(filename=ruta))

    # ══════════════════════════════════════════════════════════════════════
    # PRODUCTO 5 — Balance de área (.txt)
    # ══════════════════════════════════════════════════════════════════════
    print(f"  [5/5] Balance de área...")

    ganancia_m2, perdida_m2 = area_entre_shorelines(sl_ini, sl_fin)
    neto_m2   = ganancia_m2 - perdida_m2

    # LRR promedio de todos los transectos válidos
    lrr_mean  = np.nanmean([v for v in lrr_vals.values() if not np.isnan(v)])
    cambio_med = np.nanmedian(list(cambios_netos.values()))
    dias_periodo = (output['dates'][idx_fin] - output['dates'][idx_ini]).days
    anos_periodo = dias_periodo / 365.25

    lineas = [
        f"BALANCE DE ÁREA — {sitename}",
        f"{'='*55}",
        f"Período analizado : {f_ini} → {f_fin}",
        f"Duración           : {dias_periodo} días ({anos_periodo:.2f} años)",
        f"Imágenes válidas   : {len(idx_validos)}",
        f"Transectos válidos : {len(cols_ok)}",
        f"",
        f"CAMBIO DE LÍNEA COSTERA",
        f"{'─'*55}",
        f"Cambio neto mediano   : {cambio_med:+.2f} m",
        f"Tasa LRR promedio     : {lrr_mean:+.3f} m/año",
        f"",
        f"ÁREA ESTIMADA",
        f"{'─'*55}",
        f"Área ganada (acreción): {ganancia_m2/1e4:.2f} ha  ({ganancia_m2:.0f} m²)",
        f"Área perdida (erosión): {perdida_m2/1e4:.2f} ha  ({perdida_m2:.0f} m²)",
        f"Balance neto          : {neto_m2/1e4:+.2f} ha  ({neto_m2:+.0f} m²)",
        f"",
        f"CAMBIO POR TRANSECTO",
        f"{'─'*55}",
    ]
    for col in cols_ok:
        cn  = cambios_netos.get(col, np.nan)
        lr  = lrr_vals.get(col, np.nan)
        sig = '+' if (not np.isnan(cn) and cn >= 0) else ''
        lineas.append(
            f"  {col:<12}  neto: {sig}{cn:+6.1f} m   LRR: "
            f"{'' if np.isnan(lr) else f'{lr:+.3f}'} m/año"
        )

    lineas += [
        f"",
        f"{'─'*55}",
        f"Nota: Área estimada a partir de la geometría de las shorelines",
        f"inicial y final seleccionadas por menor ruido geométrico.",
        f"LRR = Linear Regression Rate (regresión sobre serie completa).",
    ]

    ruta_txt = os.path.join(carpeta_out, f'{sitename}_Balance_Area.txt')
    with open(ruta_txt, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lineas))
    print(f"  TXT guardado → {os.path.relpath(ruta_txt, carpeta_seg)}")

    # Imprimir resumen en consola
    print(f"\n  ── RESUMEN ──")
    print(f"  Cambio mediano : {cambio_med:+.1f} m")
    print(f"  LRR promedio   : {lrr_mean:+.3f} m/año")
    print(f"  Balance neto   : {neto_m2/1e4:+.2f} ha")
    print(f"\n  ✅  {sitename} — completado")
    print(f"      Archivos en: {os.path.relpath(carpeta_out, filepath)}/")

print(f"\n{'='*65}")
print("  ✅  TODOS LOS SEGMENTOS PROCESADOS")
print(f"{'='*65}")

DESCARGA DE DATA (TODAS LAS IMAGENES)

In [ ]:
import shutil
from google.colab import files

# 1. Comprimir toda la carpeta 'data' en un archivo ZIP
print("Comprimiendo la carpeta 'data' completa...")
shutil.make_archive('toda_mi_data', 'zip', 'data')

# 2. Descargar el archivo ZIP a tu computadora
print("Iniciando la descarga en tu computadora. Por favor espera...")
files.download('toda_mi_data.zip')


---
**DATA ANALYTIC**


---

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
from shapely.geometry import LineString
from coastsat import SDS_tools, SDS_transects

from google.colab import drive
drive.mount('/content/drive')

filepath = '/content/drive/MyDrive/coast'

# 1. Definimos los nombres de los 3 segmentos
nombres_segmentos = [
    'segmento_costa_oeste_puerto',
    'segmento_malecon_general',
    'segmento_playas_salgar_principal',
    'segmento_puerto_salgar_amplio',
    'segmento_sabanilla_limite_norte',
    'segmento_transicion_salgar_sabanilla'
]

# 2. Bucle para entrar a cada carpeta, leer el archivo guardado y sacar las gráficas
for sitename in nombres_segmentos:
    print(f"\n{'='*60}")
    print(f"📊 Generando Informe Científico Avanzado para: {sitename}")
    print(f"{'='*60}")

    carpeta = os.path.join(filepath, sitename)
    archivo_pkl = os.path.join(carpeta, f'{sitename}_output.pkl')

    if not os.path.exists(archivo_pkl):
        print(f"❌ No se encontró el archivo de datos para {sitename}. ¿Seguro que ya lo procesaste?")
        continue

    # CARGAR DATOS SIN VOLVER A DESCARGAR IMÁGENES
    with open(archivo_pkl, 'rb') as f:
        output = pickle.load(f)

    output = SDS_tools.remove_duplicates(output)
    output = SDS_tools.remove_inaccurate_georef(output, 10)

    if len(output['shorelines']) > 0:
        transects = dict([])
        idx_validos = [i for i, sl in enumerate(output['shorelines']) if len(sl) > 50]

        if len(idx_validos) > 0:
            L0 = LineString(output['shorelines'][idx_validos[0]])
            espaciado_m = 100
            longitud_mitad = 300

            # Crear transectos cada 100m
            for i, dist in enumerate(np.arange(10, L0.length, espaciado_m)):
                p = L0.interpolate(dist)
                p_antes = L0.interpolate(max(0, dist - 5))
                p_despues = L0.interpolate(min(L0.length, dist + 5))
                t = np.array([p_despues.x - p_antes.x, p_despues.y - p_antes.y])
                t = t / np.linalg.norm(t) if np.linalg.norm(t) > 0 else np.array([1,0])
                n = np.array([-t[1], t[0]])
                p1 = np.array([p.x, p.y]) - n * longitud_mitad
                p2 = np.array([p.x, p.y]) + n * longitud_mitad
                transects[f'Transecto_{i+1}'] = np.array([p1, p2])

            # Control de Calidad Base
            settings_transects = {
                'along_dist': 25, 'min_points': 3, 'max_std': 15, 'max_range': 30,
                'min_chainage': -100, 'multiple_inter': 'auto', 'auto_prc': 0.1,
            }
            # OBTENER DISTANCIAS
            cross_distance = SDS_transects.compute_intersection_QC(output, transects, settings_transects)

            # FILTRO CIENTÍFICO: Limpiar picos extremos
            settings_outliers = {
                'otsu_threshold': [-0.5, 0],
                'max_cross_change': 40,
                'plot_fig': False,
            }
            cross_distance = SDS_transects.reject_outliers(cross_distance, output, settings_outliers)

            # ¡CORRECCIÓN! Guardar CSV solo con las llaves que SOBREVIVIERON al filtro
            out_dict = {'dates': output['dates']}
            for key in cross_distance.keys():  # <--- Aquí está la magia
                out_dict[key] = cross_distance[key]

            df = pd.DataFrame(out_dict)
            ruta_csv = os.path.join(carpeta, f'{sitename}_reporte_limpio.csv')
            df.to_csv(ruta_csv, sep=',', index=False)

            # Guardamos el GeoJSON usando también solo los transectos sobrevivientes
            transects_validos = {k: v for k, v in transects.items() if k in cross_distance.keys()}
            if len(transects_validos) > 0:
                gdf = SDS_tools.transects_to_gdf(transects_validos)
                # Usamos el EPSG predeterminado de tu proyecto (32618)
                gdf.set_crs(epsg=32618, inplace=True, allow_override=True)
                ruta_geojson = os.path.join(carpeta, f'{sitename}_transectos.geojson')
                gdf.to_file(ruta_geojson, driver='GeoJSON', encoding='utf-8')

            print(f"✓ CSV de datos actualizado y guardado.")

            # ---------------------------------------------------------
            # GRÁFICOS VISUALES DEL DESPLAZAMIENTO AVANZADOS
            # ---------------------------------------------------------
            print("📈 Creando gráficos visuales de desplazamiento...")

            # A. MAPA: Primera costa vs Última costa
            fig, ax = plt.subplots(figsize=(10, 8))
            ax.axis('equal')
            ax.grid(linestyle=':', color='0.5')

            sl_inicio = output['shorelines'][idx_validos[0]]
            fecha_inicio = output['dates'][idx_validos[0]].strftime('%Y-%m-%d')
            ax.plot(sl_inicio[:, 0], sl_inicio[:, 1], 'b-', linewidth=2, label=f'Inicio ({fecha_inicio})')

            sl_fin = output['shorelines'][idx_validos[-1]]
            fecha_fin = output['dates'][idx_validos[-1]].strftime('%Y-%m-%d')
            ax.plot(sl_fin[:, 0], sl_fin[:, 1], 'r-', linewidth=2, label=f'Fin ({fecha_fin})')

            ax.set_title(f'Desplazamiento Histórico: {sitename}\n(Azul = Pasado | Rojo = Presente)', fontsize=14)
            ax.legend()
            ruta_mapa = os.path.join(carpeta, f'{sitename}_Mapa_AntesVsDespues.png')
            plt.savefig(ruta_mapa, dpi=300, bbox_inches='tight')
            plt.close(fig)
            display(Image(filename=ruta_mapa))

            # B. GRÁFICO DE LÍNEAS CON PROMEDIO MENSUAL
            # Si todas las columnas se borraron por ser malas, evitamos el error:
            if not df.drop(columns=['dates']).empty:
                tendencia_general = df.drop(columns=['dates']).mean(axis=1)
                tendencia_general = tendencia_general - tendencia_general.iloc[0]

                fechas_validas = [d for i, d in enumerate(output['dates']) if not np.isnan(tendencia_general.iloc[i])]
                tendencia_valida = tendencia_general.dropna().values

                # Promedio suavizado mensual de CoastSat
                _, dates_month, chainage_month, _ = SDS_transects.monthly_average(fechas_validas, tendencia_valida)

                fig2, ax2 = plt.subplots(figsize=(12, 5))
                ax2.grid(linestyle=':', color='0.5')

                ax2.plot(df['dates'], tendencia_general, '+', color='gray', markersize=4, alpha=0.5, label='Datos diarios (Crudos)')
                ax2.plot(dates_month, chainage_month, '-o', color='purple', linewidth=2, markersize=5, label='Tendencia Suavizada (Mensual)')

                ax2.set_title(f'Evolución de la Línea de Costa (Promedio) - {sitename}', fontsize=14)
                ax2.set_ylabel('Desplazamiento [Metros]\n(+ Crecimiento / - Erosión)', fontsize=12)
                ax2.axhline(y=0, color='black', linestyle='--', linewidth=1.5)
                ax2.legend()

                ruta_tendencia = os.path.join(carpeta, f'{sitename}_Tendencia_Erosion.png')
                plt.savefig(ruta_tendencia, dpi=300, bbox_inches='tight')
                plt.close(fig2)
                display(Image(filename=ruta_tendencia))
            else:
                print(f"⚠️ Atención: Los transectos de {sitename} tenían demasiado ruido y fueron filtrados. Revisa el polígono.")

    else:
        print(f"❌ No hay líneas de costa procesadas para {sitename}.")

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import Image, display
from shapely.geometry import LineString
from coastsat import SDS_tools, SDS_transects
from google.colab import drive

drive.mount('/content/drive')

filepath = '/content/drive/MyDrive/coast'

# 1. Definimos los nombres de los segmentos
nombres_segmentos = [
   'segmento_costa_oeste_puerto',
    'segmento_malecon_general',
    'segmento_playas_salgar_principal',
    'segmento_puerto_salgar_amplio',
    'segmento_sabanilla_limite_norte',
    'segmento_transicion_salgar_sabanilla'
]

for sitename in nombres_segmentos:
    print(f"\n{'='*60}")
    print(f"📊 Generando Mapa de Vectores de Desplazamiento para: {sitename}")
    print(f"{'='*60}")

    carpeta = os.path.join(filepath, sitename)
    archivo_pkl = os.path.join(carpeta, f'{sitename}_output.pkl')

    if not os.path.exists(archivo_pkl):
        print(f"❌ No se encontró el archivo de datos para {sitename}.")
        continue

    with open(archivo_pkl, 'rb') as f:
        output = pickle.load(f)

    output = SDS_tools.remove_duplicates(output)
    output = SDS_tools.remove_inaccurate_georef(output, 10)

    if len(output['shorelines']) == 0:
        print(f"❌ No hay líneas de costa procesadas para {sitename}.")
        continue

    transects = dict([])
    idx_validos = [i for i, sl in enumerate(output['shorelines']) if len(sl) > 50]

    if len(idx_validos) == 0:
        print(f"⚠️ No hay shorelines válidas (len>50) en {sitename}.")
        continue

    L0 = LineString(output['shorelines'][idx_validos[0]])
    espaciado_m = 100
    longitud_mitad = 300

    # Crear transectos cada 100m
    for i, dist in enumerate(np.arange(10, L0.length, espaciado_m)):
        p = L0.interpolate(dist)
        p_antes = L0.interpolate(max(0, dist - 5))
        p_despues = L0.interpolate(min(L0.length, dist + 5))
        t = np.array([p_despues.x - p_antes.x, p_despues.y - p_antes.y])
        t = t / np.linalg.norm(t) if np.linalg.norm(t) > 0 else np.array([1, 0])
        n = np.array([-t[1], t[0]])
        p1 = np.array([p.x, p.y]) - n * longitud_mitad
        p2 = np.array([p.x, p.y]) + n * longitud_mitad
        transects[f'Transecto_{i+1}'] = np.array([p1, p2])

    settings_transects = {
        'along_dist': 25, 'min_points': 3, 'max_std': 15, 'max_range': 30,
        'min_chainage': -100, 'multiple_inter': 'auto', 'auto_prc': 0.1,
    }
    cross_distance = SDS_transects.compute_intersection_QC(output, transects, settings_transects)

    settings_outliers = {
        'otsu_threshold': [-0.5, 0],
        'max_cross_change': 40,
        'plot_fig': False,
    }
    cross_distance = SDS_transects.reject_outliers(cross_distance, output, settings_outliers)

    out_dict = {'dates': output['dates']}
    for key in cross_distance.keys():
        out_dict[key] = cross_distance[key]

    df = pd.DataFrame(out_dict)

    # ---------------------------------------------------------
    # A. MAPA DE VECTORES DE DESPLAZAMIENTO (COMO LO DIBUJASTE)
    # ---------------------------------------------------------
    print("📈 Dibujando distancias en metros sobre el mapa...")

    fig, ax = plt.subplots(figsize=(12, 10))
    ax.axis('equal')
    ax.grid(linestyle=':', color='0.5')

    # Dibujar la costa más antigua (AZUL)
    sl_inicio = output['shorelines'][idx_validos[0]]
    fecha_inicio = output['dates'][idx_validos[0]].strftime('%Y-%m-%d')
    ax.plot(sl_inicio[:, 0], sl_inicio[:, 1], 'b-', linewidth=2, label=f'Costa Inicial ({fecha_inicio})')

    # Dibujar la costa más reciente (ROJA)
    sl_fin = output['shorelines'][idx_validos[-1]]
    fecha_fin = output['dates'][idx_validos[-1]].strftime('%Y-%m-%d')
    ax.plot(sl_fin[:, 0], sl_fin[:, 1], 'r-', linewidth=2, label=f'Costa Final ({fecha_fin})')

    # Dibujar los transectos, puntos y distancias
    for col in df.columns:
        if col == 'dates':
            continue

        valid_data = df[['dates', col]].dropna()
        if len(valid_data) < 2:
            continue

        dist_inicio = float(valid_data.iloc[0][col])
        dist_fin = float(valid_data.iloc[-1][col])
        cambio_neto = dist_fin - dist_inicio

        # Coordenadas del transecto
        p1 = transects[col][0]
        p2 = transects[col][1]

        vec = p2 - p1
        norm = np.linalg.norm(vec)
        if norm == 0:
            continue
        u_vec = vec / norm

        # Puntos sobre el transecto
        pt_inicio = p1 + u_vec * dist_inicio
        pt_fin = p1 + u_vec * dist_fin

        # Línea que los conecta
        ax.plot([pt_inicio[0], pt_fin[0]], [pt_inicio[1], pt_fin[1]], 'k--', linewidth=1.5, alpha=0.7)

        # Puntos
        ax.plot(pt_inicio[0], pt_inicio[1], 'ko', markersize=4)
        ax.plot(pt_fin[0], pt_fin[1], 'ko', markersize=4)

        # Texto en el punto medio
        mid = (pt_inicio + pt_fin) / 2
        color_texto = 'green' if cambio_neto >= 0 else 'red'
        signo = '+' if cambio_neto >= 0 else ''
        ax.text(mid[0], mid[1], f'{signo}{cambio_neto:.1f} m',
                color=color_texto, fontsize=10, fontweight='bold',
                ha='center', va='center',
                bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=1))

    ax.set_title(f'Vectores de Desplazamiento Neto: {sitename}\n(Metros reales de erosión o crecimiento)', fontsize=15)
    ax.set_xlabel('Eastings [EPSG:32618]')
    ax.set_ylabel('Northings [EPSG:32618]')

    # --- Leyenda combinada: líneas + explicación colores del texto ---
    ganancia = mpatches.Patch(color='green', label='Verde: Ganancia / Acreción (+ m)')
    perdida  = mpatches.Patch(color='red',   label='Rojo: Pérdida / Erosión (- m)')
    handles_lineas, labels_lineas = ax.get_legend_handles_labels()
    ax.legend(handles=handles_lineas + [ganancia, perdida],
              labels=labels_lineas + [ganancia.get_label(), perdida.get_label()],
              loc='lower left', frameon=True)

    ruta_mapa = os.path.join(carpeta, f'{sitename}_Mapa_Vectores_Metros.png')
    plt.savefig(ruta_mapa, dpi=300, bbox_inches='tight')
    plt.close(fig)
    display(Image(filename=ruta_mapa))

    # ---------------------------------------------------------
    # B. GRÁFICO DE LÍNEAS CON PROMEDIO MENSUAL
    # ---------------------------------------------------------
    if not df.drop(columns=['dates']).empty:
        tendencia_general = df.drop(columns=['dates']).mean(axis=1)
        tendencia_general = tendencia_general - tendencia_general.iloc[0]

        fechas_validas = [d for i, d in enumerate(output['dates']) if not np.isnan(tendencia_general.iloc[i])]
        tendencia_valida = tendencia_general.dropna().values

        _, dates_month, chainage_month, _ = SDS_transects.monthly_average(fechas_validas, tendencia_valida)

        fig2, ax2 = plt.subplots(figsize=(12, 5))
        ax2.grid(linestyle=':', color='0.5')

        ax2.plot(df['dates'], tendencia_general, '+', color='gray', markersize=4, alpha=0.5, label='Datos diarios (Crudos)')
        ax2.plot(dates_month, chainage_month, '-o', color='purple', linewidth=2, markersize=5, label='Tendencia Suavizada (Mensual)')

        ax2.set_title(f'Evolución de la Línea de Costa (Promedio) - {sitename}', fontsize=14)
        ax2.set_ylabel('Desplazamiento [Metros]\n(+ Crecimiento / - Erosión)', fontsize=12)
        ax2.axhline(y=0, color='black', linestyle='--', linewidth=1.5)
        ax2.legend()

        ruta_tendencia = os.path.join(carpeta, f'{sitename}_Tendencia_Erosion.png')
        plt.savefig(ruta_tendencia, dpi=300, bbox_inches='tight')
        plt.close(fig2)
        display(Image(filename=ruta_tendencia))
    else:
        print(f"⚠️ Atención: Los transectos de {sitename} tenían demasiado ruido y fueron filtrados.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Mapas de Vectores + Series Temporales — COSTAS_GITT
# Solo genera gráficas. Lee PKLs ya descargados de "analisis abril".
# NO descarga imágenes nuevas ni genera JSON.
# ══════════════════════════════════════════════════════════════════════════════

import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
from IPython.display import Image, display
from shapely.geometry import LineString
from coastsat import SDS_tools, SDS_transects
from google.colab import drive
import warnings
warnings.filterwarnings('ignore')

# ── CONFIG ────────────────────────────────────────────────────────────────────
drive.mount('/content/drive')

# ✅ Subcarpeta donde están los PKLs
filepath = '/content/drive/MyDrive/coast'

nombres_segmentos = [
    'segmento_costa_oeste_puerto',
    'segmento_malecon_general',
    'segmento_playas_salgar_principal',
    'segmento_puerto_salgar_amplio',
    'segmento_sabanilla_limite_norte',
    'segmento_transicion_salgar_sabanilla',
]

# ── Parámetros de transectos ──────────────────────────────────────────────────
espaciado_m    = 300
longitud_mitad = 300
min_len_sl     = 80

# ── QC estricto (valores recomendados CoastSat para Caribe microtidal) ────────
settings_transects = {
    'along_dist':    25,
    'min_points':    3,
    'max_std':       15,
    'max_range':     30,
    'min_chainage': -100,
    'multiple_inter': 'auto',
    'auto_prc':      0.1,
}
settings_outliers = {
    'otsu_threshold':  [-0.5, 0],
    'max_cross_change': 20,     # 20 m entre fechas consecutivas
    'plot_fig':        False,
}

# ── Umbrales de visualización ─────────────────────────────────────────────────
UMBRAL_ETIQUETA_M = 5     # etiqueta solo si |cambio| >= 5 m (evita ruido visual)
UMBRAL_DIBUJO_M   = 200   # descarta transecto del mapa si |cambio| > 200 m

# ── Suavizado ─────────────────────────────────────────────────────────────────
ROLLING_DIAS = 60   # ventana de mediana móvil
PCTL_LO, PCTL_HI = 10, 90   # recorte inter-transecto


# ══════════════════════════════════════════════════════════════════════════════
# HELPERS
# ══════════════════════════════════════════════════════════════════════════════
def shoreline_valida(sl, max_jump_factor=10):
    """Descarta shorelines con geometría rota (loops, saltos extremos)."""
    if sl is None or len(sl) < min_len_sl:
        return False
    dif  = np.diff(sl, axis=0)
    step = np.sqrt(dif[:, 0]**2 + dif[:, 1]**2)
    if len(step) == 0:
        return False
    med = np.median(step)
    if med <= 0:
        return False
    return np.max(step) <= max_jump_factor * med


def elegir_extremos(output, idx_validos, frac=0.15):
    """
    Elige las shorelines de inicio y fin por menor ruido interno,
    dentro del primer/último 15% del período — no por posición temporal.
    Evita que una shoreline con nubes residuales defina los vectores.
    """
    def ruido(sl):
        dif  = np.diff(sl, axis=0)
        step = np.sqrt(dif[:, 0]**2 + dif[:, 1]**2)
        return float(np.std(step)) if len(step) > 0 else 9e9

    n     = len(idx_validos)
    corte = max(1, int(n * frac))
    idx_ini = min(idx_validos[:corte],  key=lambda i: ruido(output['shorelines'][i]))
    idx_fin = min(idx_validos[-corte:], key=lambda i: ruido(output['shorelines'][i]))
    return idx_ini, idx_fin


def promedio_robusto(df, cols):
    """
    Recorta P10–P90 entre transectos en cada fecha antes de promediar.
    Un transecto con outlier residual ya no arrastra la media de toda la fila.
    """
    X  = df[cols].copy()
    lo = X.quantile(PCTL_LO / 100.0, axis=1)
    hi = X.quantile(PCTL_HI / 100.0, axis=1)
    for c in cols:
        X[c] = X[c].where((X[c] >= lo) & (X[c] <= hi))
    return X.mean(axis=1, skipna=True)


# ══════════════════════════════════════════════════════════════════════════════
# BUCLE PRINCIPAL
# ══════════════════════════════════════════════════════════════════════════════
for sitename in nombres_segmentos:
    print(f"\n{'='*65}")
    print(f"  {sitename}")
    print(f"{'='*65}")

    # ── Localizar el PKL — orden de prioridad según diagnóstico ─────────
    # 1. Analisis Abril/_output.pkl  (el más completo: 1.3–2.8 MB)
    # 2. _output_merged.pkl en la raíz  (0.1 MB, puede ser parcial)
    # 3. Analisis Mayo/_output.pkl
    # 4. raíz/_output.pkl
    candidatos = [
        os.path.join(filepath, sitename, 'Analisis Abril', f'{sitename}_output.pkl'),
        os.path.join(filepath, sitename, f'{sitename}_output_merged.pkl'),
        os.path.join(filepath, sitename, 'Analisis Mayo',  f'{sitename}_output.pkl'),
        os.path.join(filepath, sitename, f'{sitename}_output.pkl'),
    ]
    etiquetas = ['Analisis Abril', 'raiz (merged)', 'Analisis Mayo', 'raiz']

    pkl_path = None
    carpeta  = os.path.join(filepath, sitename, 'Analisis Abril')  # donde guardar graficas
    for ruta, etiq in zip(candidatos, etiquetas):
        if os.path.exists(ruta):
            pkl_path = ruta
            carpeta  = os.path.dirname(ruta)
            print(f"  PKL: {etiq} -> {os.path.basename(ruta)}")
            break

    if not pkl_path:
        print(f"  ❌  Ningún PKL encontrado.")
        continue

    with open(pkl_path, 'rb') as f:
        output = pickle.load(f)

    output = SDS_tools.remove_duplicates(output)
    output = SDS_tools.remove_inaccurate_georef(output, 10)

    if not output.get('shorelines'):
        print(f"  ❌  Sin shorelines en el PKL.")
        continue

    print(f"  🗓️  {output['dates'][0].strftime('%Y-%m-%d')} → {output['dates'][-1].strftime('%Y-%m-%d')}")
    print(f"      {len(output['shorelines'])} shorelines totales")

    # ── 1. Filtrar shorelines válidas ─────────────────────────────────────
    idx_validos = sorted(
        [i for i, sl in enumerate(output['shorelines']) if shoreline_valida(sl)],
        key=lambda i: output['dates'][i]
    )
    if len(idx_validos) < 2:
        print(f"  ⚠️  Menos de 2 shorelines válidas tras filtro de geometría.")
        continue
    print(f"      {len(idx_validos)} shorelines válidas")

    # ── 2. Extremos por calidad (no por posición temporal) ────────────────
    idx_ini, idx_fin = elegir_extremos(output, idx_validos)

    # ── 3. Transectos sobre la shoreline CENTRAL ──────────────────────────
    idx_base = idx_validos[len(idx_validos) // 2]
    L0 = LineString(output['shorelines'][idx_base])
    if L0.length <= 0:
        print(f"  ⚠️  Shoreline base sin longitud.")
        continue

    transects = {}
    for i, dist in enumerate(np.arange(10, L0.length, espaciado_m)):
        p       = L0.interpolate(dist)
        p_antes = L0.interpolate(max(0, dist - 5))
        p_desp  = L0.interpolate(min(L0.length, dist + 5))
        t = np.array([p_desp.x - p_antes.x, p_desp.y - p_antes.y])
        nrm = np.linalg.norm(t)
        t = t / nrm if nrm > 0 else np.array([1.0, 0.0])
        n = np.array([-t[1], t[0]])
        p1 = np.array([p.x, p.y]) - n * longitud_mitad
        p2 = np.array([p.x, p.y]) + n * longitud_mitad
        transects[f'T_{i+1}'] = np.array([p1, p2])

    print(f"  📍  {len(transects)} transectos generados")

    # ── 4. Intersecciones QC + reject outliers ────────────────────────────
    cross = SDS_transects.compute_intersection_QC(output, transects, settings_transects)
    if not cross:
        print(f"  ⚠️  Sin datos tras QC.")
        continue
    cross = SDS_transects.reject_outliers(cross, output, settings_outliers)
    if not cross:
        print(f"  ⚠️  Sin datos tras reject_outliers.")
        continue

    df = pd.DataFrame({'dates': output['dates']})
    for k, v in cross.items():
        df[k] = v

    # ── 5. Descartar transectos con cambio neto absurdo ───────────────────
    cols_ok = [
        k for k in cross
        if len(df[['dates', k]].dropna()) >= 2
        and abs(float(df[k].dropna().iloc[-1]) - float(df[k].dropna().iloc[0])) <= UMBRAL_DIBUJO_M
    ]
    if not cols_ok:
        print(f"  ⚠️  Todos los transectos superan {UMBRAL_DIBUJO_M} m — revisa los datos.")
        continue
    print(f"      {len(cols_ok)} transectos válidos para graficar")

    # ── 6. Promedio robusto + centrado por mediana del primer 10% ─────────
    tendencia = promedio_robusto(df, cols_ok)
    n_ref = max(1, int(len(tendencia.dropna()) * 0.10))
    ref   = float(tendencia.dropna().iloc[:n_ref].median())
    tendencia = tendencia - ref

    # ── 7. Rolling median CENTRADA ────────────────────────────────────────
    df_s   = pd.DataFrame({'val': tendencia.values}, index=pd.to_datetime(df['dates']))
    df_s   = df_s.sort_index()
    smooth = df_s['val'].rolling(f'{ROLLING_DIAS}D', min_periods=3, center=True).median()

    # ── Carpeta de salida (misma que el PKL) ──────────────────────────────
    carpeta_out = carpeta

    # ══════════════════════════════════════════════════════════════════════
    # GRÁFICO A — Mapa de vectores de desplazamiento neto
    # ══════════════════════════════════════════════════════════════════════
    fig, ax = plt.subplots(figsize=(13, 10))
    ax.axis('equal')
    ax.set_facecolor('#f8f9fa')
    ax.grid(linestyle=':', color='0.7', alpha=0.6)

    sl_ini   = output['shorelines'][idx_ini]
    sl_fin   = output['shorelines'][idx_fin]
    f_ini    = output['dates'][idx_ini].strftime('%Y-%m-%d')
    f_fin    = output['dates'][idx_fin].strftime('%Y-%m-%d')

    ax.plot(sl_ini[:, 0], sl_ini[:, 1], '-', color='steelblue',  lw=2.2, label=f'Costa inicial ({f_ini})', zorder=4)
    ax.plot(sl_fin[:, 0], sl_fin[:, 1], '-', color='firebrick',  lw=2.2, label=f'Costa final   ({f_fin})',  zorder=4)

    n_acrecion = 0
    n_erosion  = 0

    for col in cols_ok:
        valido = df[['dates', col]].dropna()
        if len(valido) < 2:
            continue
        d0      = float(valido[col].iloc[0])
        d1      = float(valido[col].iloc[-1])
        cambio  = d1 - d0

        p1, p2  = transects[col][0], transects[col][1]
        vec     = p2 - p1
        nrm     = np.linalg.norm(vec)
        if nrm == 0:
            continue
        u   = vec / nrm
        pt0 = p1 + u * d0
        pt1 = p1 + u * d1

        color_v = '#2d9e2d' if cambio >= 0 else '#c0392b'
        alpha_v = 0.35 + min(abs(cambio) / UMBRAL_DIBUJO_M, 0.65)

        ax.annotate('', xy=(pt1[0], pt1[1]), xytext=(pt0[0], pt0[1]),
                    arrowprops=dict(arrowstyle='->', color=color_v,
                                   lw=1.8, alpha=alpha_v),
                    zorder=3)
        ax.plot(pt0[0], pt0[1], 'o', color=color_v, ms=3.5, alpha=0.7, zorder=3)

        if abs(cambio) >= UMBRAL_ETIQUETA_M:
            mid   = (pt0 + pt1) / 2
            signo = '+' if cambio >= 0 else ''
            ax.text(mid[0], mid[1], f'{signo}{cambio:.1f} m',
                    color=color_v, fontsize=9, fontweight='bold',
                    ha='center', va='center', zorder=5,
                    bbox=dict(facecolor='white', alpha=0.82,
                              edgecolor='none', pad=1.5))

        if cambio >= 0:
            n_acrecion += 1
        else:
            n_erosion  += 1

    ax.set_title(
        f'Vectores de desplazamiento neto — {sitename}\n'
        f'Período: {f_ini} → {f_fin}',
        fontsize=13, fontweight='bold', pad=14
    )
    ax.set_xlabel('Eastings [EPSG:32618]', fontsize=10)
    ax.set_ylabel('Northings [EPSG:32618]', fontsize=10)

    g_p = mpatches.Patch(color='#2d9e2d', label=f'Ganancia / Acreción (+m)  — {n_acrecion} transectos')
    r_p = mpatches.Patch(color='#c0392b', label=f'Pérdida / Erosión (−m)   — {n_erosion} transectos')
    h, l = ax.get_legend_handles_labels()
    ax.legend(handles=h + [g_p, r_p], labels=l + [g_p.get_label(), r_p.get_label()],
              loc='lower left', frameon=True, fontsize=9, framealpha=0.9)

    plt.tight_layout()
    ruta_A = os.path.join(carpeta_out, f'{sitename}_Mapa_Vectores_Metros.png')
    plt.savefig(ruta_A, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"  💾  Mapa vectores → {ruta_A}")
    display(Image(filename=ruta_A))

    # ══════════════════════════════════════════════════════════════════════
    # GRÁFICO B — Serie temporal con rolling median centrada
    # ══════════════════════════════════════════════════════════════════════
    fig2, ax2 = plt.subplots(figsize=(13, 5))
    ax2.set_facecolor('#f8f9fa')
    ax2.grid(linestyle=':', color='0.7', alpha=0.5)

    # Banda ±1σ entre transectos (muestra variabilidad espacial real)
    std_serie = df[cols_ok].std(axis=1, skipna=True)
    ax2.fill_between(
        df_s.index,
        df_s['val'] - std_serie.values,
        df_s['val'] + std_serie.values,
        color='steelblue', alpha=0.12,
        label='±1σ variabilidad entre transectos'
    )

    # Sombreado de regiones erosión/acreción bajo la curva suavizada
    ax2.fill_between(smooth.index, 0, smooth,
                     where=(smooth >= 0),
                     color='#2d9e2d', alpha=0.18, label='Acreción (suavizado)')
    ax2.fill_between(smooth.index, 0, smooth,
                     where=(smooth < 0),
                     color='#c0392b', alpha=0.18, label='Erosión (suavizado)')

    # Puntos crudos
    ax2.plot(df_s.index, df_s['val'],
             '+', color='#888', ms=4, alpha=0.4,
             label='Datos filtrados (por fecha)')

    # Mediana móvil centrada
    ax2.plot(smooth.index, smooth,
             '-', color='#4c1d95', lw=2.5,
             label=f'Mediana móvil {ROLLING_DIAS}d (centrada)')

    ax2.axhline(0, color='black', ls='--', lw=1.2, alpha=0.6)

    # Eje X formateado por año
    ax2.xaxis.set_major_locator(mdates.YearLocator())
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax2.xaxis.set_minor_locator(mdates.MonthLocator(bymonth=[4, 7, 10]))
    plt.setp(ax2.xaxis.get_majorticklabels(), rotation=0, ha='center')

    # Texto con delta total
    if smooth.dropna().shape[0] >= 2:
        delta_tot = float(smooth.dropna().iloc[-1]) - float(smooth.dropna().iloc[0])
        signo_d   = '+' if delta_tot >= 0 else ''
        color_d   = '#2d9e2d' if delta_tot >= 0 else '#c0392b'
        ax2.text(0.015, 0.97,
                 f'Δ total: {signo_d}{delta_tot:.1f} m  ({f_ini} → {f_fin})',
                 transform=ax2.transAxes, fontsize=9.5,
                 verticalalignment='top', color=color_d, fontweight='bold',
                 bbox=dict(facecolor='white', alpha=0.85, edgecolor='none', pad=3))

    ax2.set_title(f'Evolución de la línea de costa — {sitename}', fontsize=13, fontweight='bold')
    ax2.set_ylabel('Desplazamiento [m]\n(+ acreción  /  − erosión)', fontsize=10)
    ax2.legend(loc='lower left', fontsize=8.5, framealpha=0.9)

    plt.tight_layout()
    ruta_B = os.path.join(carpeta_out, f'{sitename}_Tendencia_Erosion.png')
    plt.savefig(ruta_B, dpi=300, bbox_inches='tight')
    plt.close(fig2)
    print(f"  💾  Serie temporal  → {ruta_B}")
    display(Image(filename=ruta_B))

    print(f"\n  ✅  {sitename} — completado")

print(f"\n{'='*65}")
print("  ✅  TODOS LOS SEGMENTOS PROCESADOS")
print(f"{'='*65}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# GENERAR PKL CON RANGO DE FECHAS PERSONALIZADO
# ──────────────────────────────────────────────────────────────────────────────
# Las imágenes ya están descargadas en Drive.
# Este script filtra las que caen en el rango que elijas y genera un PKL nuevo
# SIN tocar GEE ni descargar nada.
# ══════════════════════════════════════════════════════════════════════════════

import os
import pickle
import numpy as np
from datetime import datetime
from google.colab import drive
from coastsat import SDS_shoreline, SDS_tools, SDS_preprocess

drive.mount('/content/drive')

filepath = '/content/drive/MyDrive/coast'

# ══════════════════════════════════════════════════════════════════════════════
# ▶▶  CONFIGURA AQUÍ EL RANGO QUE QUIERES  ◀◀
# ══════════════════════════════════════════════════════════════════════════════
FECHA_INICIO = '2024-01-01'   # formato YYYY-MM-DD
FECHA_FIN    = '2026-06-04'   # hasta hoy o la fecha que necesites
NOMBRE_SALIDA = 'analisis_custom'   # nombre de la subcarpeta donde se guarda el PKL
# ══════════════════════════════════════════════════════════════════════════════

nombres_segmentos = [
    'segmento_costa_oeste_puerto',
    'segmento_malecon_general',
    'segmento_playas_salgar_principal',
    'segmento_puerto_salgar_amplio',
    'segmento_sabanilla_limite_norte',
    'segmento_transicion_salgar_sabanilla',
]

# Settings de extracción — mismos que usaste originalmente
settings = {
    'cloud_thresh':     0.10,
    'dist_clouds':      300,
    'output_epsg':      32618,
    'check_detection':  False,   # False = no abre ventana manual
    'adjust_detection': False,
    'save_figure':      False,
    'min_beach_area':   1000,
    'min_length_sl':    800,
    'cloud_mask_issue': False,
    'sand_color':       'default',
    'pan_off':          False,
    's2cloudless_prob': 60,
}

# ─────────────────────────────────────────────────────────────────────────────

from datetime import timezone
# Las fechas del PKL de CoastSat vienen con timezone UTC (offset-aware).
# Agregamos tzinfo=UTC para poder compararlas directamente.
fecha_ini_dt = datetime.strptime(FECHA_INICIO, '%Y-%m-%d').replace(tzinfo=timezone.utc)
fecha_fin_dt = datetime.strptime(FECHA_FIN,    '%Y-%m-%d').replace(tzinfo=timezone.utc)

for sitename in nombres_segmentos:
    print(f"\n{'='*60}")
    print(f"  {sitename}")
    print(f"  Rango: {FECHA_INICIO} → {FECHA_FIN}")
    print(f"{'='*60}")

    carpeta_seg = os.path.join(filepath, sitename)

    # ── 1. Cargar el PKL más completo que ya tienes ───────────────────────
    candidatos = [
        os.path.join(carpeta_seg, 'Analisis Abril', f'{sitename}_output.pkl'),
        os.path.join(carpeta_seg, f'{sitename}_output_merged.pkl'),
        os.path.join(carpeta_seg, f'{sitename}_output.pkl'),
    ]
    pkl_base = next((p for p in candidatos if os.path.exists(p)), None)

    if not pkl_base:
        print(f"  ❌  No hay PKL base. Necesitas correr la extracción inicial.")
        continue

    print(f"  📦  Base: {os.path.relpath(pkl_base, carpeta_seg)}")

    with open(pkl_base, 'rb') as f:
        output_completo = pickle.load(f)

    output_completo = SDS_tools.remove_duplicates(output_completo)
    output_completo = SDS_tools.remove_inaccurate_georef(output_completo, 10)

    total = len(output_completo.get('dates', []))
    print(f"  🗓️  PKL base: {total} fechas  "
          f"({output_completo['dates'][0].strftime('%Y-%m-%d')} → "
          f"{output_completo['dates'][-1].strftime('%Y-%m-%d')})")

    # ── 2. Filtrar por rango de fechas ────────────────────────────────────
    indices = [
        i for i, d in enumerate(output_completo['dates'])
        if fecha_ini_dt <= d <= fecha_fin_dt
    ]

    if not indices:
        print(f"  ⚠️  Ninguna fecha del PKL cae en el rango solicitado.")
        print(f"      Fechas disponibles: "
              f"{output_completo['dates'][0].strftime('%Y-%m-%d')} → "
              f"{output_completo['dates'][-1].strftime('%Y-%m-%d')}")
        continue

    print(f"  ✂️   {len(indices)} fechas dentro del rango (de {total} totales)")

    # ── 3. Construir el PKL filtrado ──────────────────────────────────────
    # Copiar todas las claves del output original, filtrando por índice
    # las que son listas indexadas por fecha.
    CLAVES_INDEXADAS = [
        'dates', 'shorelines', 'filename', 'cloud_cover',
        'idx', 'Esat', 'Nsat',      # metadatos de posición del satélite
        'geoaccuracy',              # error de georreferenciación
    ]

    output_filtrado = {}
    for clave, valor in output_completo.items():
        if clave in CLAVES_INDEXADAS and isinstance(valor, (list, np.ndarray)):
            try:
                if isinstance(valor, np.ndarray):
                    output_filtrado[clave] = valor[indices]
                else:
                    output_filtrado[clave] = [valor[i] for i in indices]
            except (IndexError, TypeError):
                # Si la clave existe pero no es indexable de esa forma, copiar tal cual
                output_filtrado[clave] = valor
        else:
            # Metadatos que no son por fecha (bbox, epsg, inputs, etc.)
            output_filtrado[clave] = valor

    # ── 4. Guardar el PKL nuevo ───────────────────────────────────────────
    carpeta_salida = os.path.join(carpeta_seg, NOMBRE_SALIDA)
    os.makedirs(carpeta_salida, exist_ok=True)

    nombre_pkl = f'{sitename}_{FECHA_INICIO[:7].replace("-","")}_{FECHA_FIN[:7].replace("-","")}_output.pkl'
    ruta_pkl   = os.path.join(carpeta_salida, nombre_pkl)

    with open(ruta_pkl, 'wb') as f:
        pickle.dump(output_filtrado, f)

    print(f"  ✅  PKL guardado: {NOMBRE_SALIDA}/{nombre_pkl}")
    print(f"      Fechas: {output_filtrado['dates'][0].strftime('%Y-%m-%d')} → "
          f"{output_filtrado['dates'][-1].strftime('%Y-%m-%d')}")
    print(f"      Shorelines válidas: {sum(1 for s in output_filtrado['shorelines'] if s is not None and len(s) > 0)}")

print(f"\n{'='*60}")
print(f"  ✅  Listo. PKLs guardados en subcarpeta '{NOMBRE_SALIDA}/'")
print(f"  Para analizar, cambia la ruta en analisis_graficas.py a:")
print(f"  os.path.join(filepath, sitename, '{NOMBRE_SALIDA}', ...)")
print(f"{'='*60}")

ANALYSIS DATA ANTERIOR

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.stats import linregress
from IPython.display import Image, display
from shapely.geometry import LineString
from coastsat import SDS_tools, SDS_transects

# 1. Configuración de rutas y segmentos
filepath = os.path.join(os.getcwd(), 'data')
nombres_segmentos = [
    'segmento_costa_oeste_puerto',
    'segmento_malecon_general',
    'segmento_playas_salgar_principal',
    'segmento_puerto_salgar_amplio',
    'segmento_sabanilla_limite_norte',
    'segmento_transicion_salgar_sabanilla'
]

espaciado_m = 100 # Separación entre transectos (metros)

for sitename in nombres_segmentos:
    print(f"\n{'='*70}")
    print(f"🏢 GENERANDO REPORTE EJECUTIVO INTEGRAL PARA: {sitename}")
    print(f"{'='*70}")

    carpeta = os.path.join(filepath, sitename)
    archivo_pkl = os.path.join(carpeta, f'{sitename}_output.pkl')

    # CREAR LA NUEVA CARPETA PARA EL REPORTE
    carpeta_reporte = os.path.join(carpeta, 'Reporte_Ejecutivo')
    os.makedirs(carpeta_reporte, exist_ok=True)

    if not os.path.exists(archivo_pkl):
        print(f"❌ No se encontró el archivo de datos para {sitename}.")
        continue

    # CARGAR Y LIMPIAR DATOS
    with open(archivo_pkl, 'rb') as f:
        output = pickle.load(f)

    output = SDS_tools.remove_duplicates(output)
    output = SDS_tools.remove_inaccurate_georef(output, 10)

    if len(output['shorelines']) > 0:
        transects = dict([])
        idx_validos = [i for i, sl in enumerate(output['shorelines']) if len(sl) > 50]

        if len(idx_validos) > 0:
            L0 = LineString(output['shorelines'][idx_validos[0]])
            longitud_mitad = 300

            # Crear transectos cada 100m
            for i, dist in enumerate(np.arange(10, L0.length, espaciado_m)):
                p = L0.interpolate(dist)
                p_antes = L0.interpolate(max(0, dist - 5))
                p_despues = L0.interpolate(min(L0.length, dist + 5))
                t = np.array([p_despues.x - p_antes.x, p_despues.y - p_antes.y])
                t = t / np.linalg.norm(t) if np.linalg.norm(t) > 0 else np.array([1,0])
                n = np.array([-t[1], t[0]])
                p1 = np.array([p.x, p.y]) - n * longitud_mitad
                p2 = np.array([p.x, p.y]) + n * longitud_mitad
                transects[f'Transecto_{i+1}'] = np.array([p1, p2])

            settings_transects = {
                'along_dist': 25, 'min_points': 3, 'max_std': 15, 'max_range': 30,
                'min_chainage': -100, 'multiple_inter': 'auto', 'auto_prc': 0.1,
            }
            cross_distance = SDS_transects.compute_intersection_QC(output, transects, settings_transects)

            settings_outliers = {
                'otsu_threshold': [-0.5, 0], 'max_cross_change': 40, 'plot_fig': False,
            }
            cross_distance = SDS_transects.reject_outliers(cross_distance, output, settings_outliers)

            # Crear DataFrame
            out_dict = {'dates': output['dates']}
            for key in cross_distance.keys():
                out_dict[key] = cross_distance[key]
            df = pd.DataFrame(out_dict)

            if df.drop(columns=['dates']).empty:
                print(f"⚠️ Sin datos válidos para {sitename}.")
                continue

            # ====================================================================
            # 📊 CÁLCULOS ESTADÍSTICOS (LRR, Cambio Neto y Área)
            # ====================================================================
            resultados_transectos = []
            area_total_ganada = 0
            area_total_perdida = 0

            for col in df.columns:
                if col == 'dates': continue
                valid_data = df[['dates', col]].dropna()
                if len(valid_data) > 2:
                    fechas = valid_data['dates'].values
                    distancias = valid_data[col].values

                    # Calcular años transcurridos para regresión
                    dias_transcurridos = np.array([(d - fechas[0]).astype('timedelta64[D]').astype(int) for d in fechas])
                    anios = dias_transcurridos / 365.25

                    # Tasa de Regresión Lineal (LRR - metros/año)
                    slope, intercept, r_value, p_value, std_err = linregress(anios, distancias)

                    # Cambio Neto (usando la tendencia para evitar ruido del último día)
                    cambio_neto = slope * anios[-1]

                    # Área (Cambio neto * Ancho de influencia del transecto, que es 100m)
                    area_m2 = cambio_neto * espaciado_m

                    if area_m2 > 0: area_total_ganada += area_m2
                    else: area_total_perdida += abs(area_m2)

                    resultados_transectos.append({
                        'Transecto': col,
                        'LRR (m/año)': slope,
                        'Cambio Neto (m)': cambio_neto,
                        'Area (m2)': area_m2
                    })

            df_resultados = pd.DataFrame(resultados_transectos)

            # Guardar reporte de texto con el Balance de Área
            balance_neto = area_total_ganada - area_total_perdida
            ruta_txt = os.path.join(carpeta_reporte, f'1_Balance_Area_{sitename}.txt')
            with open(ruta_txt, 'w') as f:
                f.write(f"REPORTE EJECUTIVO DE BALANCE SEDIMENTARIO - {sitename}\n")
                f.write("="*60 + "\n")
                f.write(f"Área Total de Playa GANADA (Acumulación): +{area_total_ganada:,.1f} m²\n")
                f.write(f"Área Total de Playa PERDIDA (Erosión):    -{area_total_perdida:,.1f} m²\n")
                f.write("-"*60 + "\n")
                if balance_neto > 0:
                    f.write(f"BALANCE NETO: La playa CRECIÓ un total de {abs(balance_neto):,.1f} m²\n")
                else:
                    f.write(f"BALANCE NETO: La playa se EROSIONÓ un total de {abs(balance_neto):,.1f} m²\n")
            print(f"✓ Balance de Área calculado y guardado.")

            # ====================================================================
            # 📊 GRÁFICO 1: PERFIL LONGITUDINAL (BARRAS DE CAMBIO NETO)
            # ====================================================================
            fig1, ax1 = plt.subplots(figsize=(14, 6))
            colores_barras = ['green' if x >= 0 else 'red' for x in df_resultados['Cambio Neto (m)']]
            nombres_eje_x = [t.replace('Transecto_', 'T') for t in df_resultados['Transecto']]

            ax1.bar(nombres_eje_x, df_resultados['Cambio Neto (m)'], color=colores_barras, alpha=0.7, edgecolor='black')
            ax1.axhline(0, color='black', linewidth=1.5)
            ax1.set_title(f'Perfil Longitudinal: Cambio Neto de la Playa - {sitename}', fontsize=16)
            ax1.set_ylabel('Cambio Total [Metros]\n(+ Crecimiento / - Erosión)', fontsize=12)
            ax1.set_xlabel('Transectos (Norte a Sur)', fontsize=12)
            plt.xticks(rotation=45)
            ax1.grid(axis='y', linestyle='--', alpha=0.7)

            ruta_perfil = os.path.join(carpeta_reporte, f'2_Perfil_Longitudinal_Barras.png')
            plt.savefig(ruta_perfil, dpi=300, bbox_inches='tight')
            plt.close(fig1)

            # ====================================================================
            # 📊 GRÁFICO 2: TASA DE CAMBIO ANUAL (LRR)
            # ====================================================================
            fig2, ax2 = plt.subplots(figsize=(14, 4))
            ax2.plot(nombres_eje_x, df_resultados['LRR (m/año)'], '-o', color='blue', linewidth=2, markersize=6)
            ax2.fill_between(nombres_eje_x, 0, df_resultados['LRR (m/año)'],
                             where=(df_resultados['LRR (m/año)'] >= 0), color='green', alpha=0.3, interpolate=True)
            ax2.fill_between(nombres_eje_x, 0, df_resultados['LRR (m/año)'],
                             where=(df_resultados['LRR (m/año)'] <= 0), color='red', alpha=0.3, interpolate=True)
            ax2.axhline(0, color='black', linewidth=1.5)
            ax2.set_title(f'Velocidad de Erosión/Acreción Anual (LRR) - {sitename}', fontsize=16)
            ax2.set_ylabel('Tasa [Metros / Año]', fontsize=12)
            plt.xticks(rotation=45)
            ax2.grid(linestyle='--', alpha=0.7)

            ruta_lrr = os.path.join(carpeta_reporte, f'3_Tasa_Anual_LRR.png')
            plt.savefig(ruta_lrr, dpi=300, bbox_inches='tight')
            plt.close(fig2)

            # ====================================================================
            # 📊 GRÁFICO 3: MAPA DE CALOR ESPACIO-TEMPORAL (HOVMÖLLER)
            # ====================================================================
            # Centrar los datos respecto al primer día de medición
            df_centered = df.copy()
            for col in df_resultados['Transecto']:
                primer_valor = df[col].dropna().iloc[0]
                df_centered[col] = df[col] - primer_valor

            # Crear matriz pivote (Filas: Transectos, Columnas: Fechas)
            df_heatmap = df_centered[df_resultados['Transecto']].T
            df_heatmap.columns = df_centered['dates']

            # Rellenar huecos (NaNs) interpolando temporalmente para que el mapa no tenga blancos
            df_heatmap = df_heatmap.interpolate(axis=1, limit_direction='both')

            fig3, ax3 = plt.subplots(figsize=(14, 8))
            # Usar mapa de colores divergente: Rojo (erosión) - Blanco (Cero) - Verde (Acreción)
            cax = ax3.pcolormesh(df_heatmap.columns, np.arange(len(df_heatmap.index)), df_heatmap.values,
                                 cmap='RdYlGn', shading='auto', vmin=-30, vmax=30)

            ax3.set_yticks(np.arange(len(df_heatmap.index)))
            ax3.set_yticklabels(nombres_eje_x)
            ax3.invert_yaxis() # Para que el Transecto 1 quede arriba (Norte)

            # Formatear el eje de fechas
            ax3.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
            plt.xticks(rotation=45)

            cbar = fig3.colorbar(cax, ax=ax3, pad=0.02)
            cbar.set_label('Desplazamiento de Costa [metros]\n(Rojo = Erosión, Verde = Crece)', fontsize=11)

            ax3.set_title(f'Mapa de Calor Espacio-Temporal (Dinámica de Arena) - {sitename}', fontsize=16)
            ax3.set_xlabel('Tiempo (Años)', fontsize=12)
            ax3.set_ylabel('Transectos (Norte a Sur)', fontsize=12)

            ruta_heat = os.path.join(carpeta_reporte, f'4_Heatmap_EspacioTemporal.png')
            plt.savefig(ruta_heat, dpi=300, bbox_inches='tight')
            plt.close(fig3)

            print(f"✓ Todos los gráficos generados y guardados en la carpeta 'Reporte_Ejecutivo'.")

            # Opcional: Mostrar uno de los gráficos de resumen aquí en Colab
            display(Image(filename=ruta_perfil))

    else:
        print(f"❌ No hay líneas de costa procesadas para {sitename}.")

mejoras codigo

In [ ]:
import numpy as np

def elegir_indices_inicio_fin(output, idx_validos, frac_ventana=0.2):
    """
    Elige indices robustos para 'inicio' y 'fin' dentro de una ventana temporal:
    - inicio: mejor calidad en el primer frac_ventana del periodo
    - fin:    mejor calidad en el último frac_ventana del periodo

    Criterio de calidad:
    - si existe output['cloud_cover']: prioriza menor nube
    - si existe output['MNDWI_threshold']: prioriza umbral cercano a la mediana (estable)
    - siempre penaliza shorelines muy cortas (pocos puntos)
    """
    # ordenar idx_validos por fecha
    idx_validos = sorted(idx_validos, key=lambda i: output['dates'][i])
    n = len(idx_validos)
    if n == 0:
        return None, None

    k = max(1, int(np.ceil(n * frac_ventana)))
    ventana_ini = idx_validos[:k]
    ventana_fin = idx_validos[-k:]

    # preparar arrays de métricas (si existen)
    has_cloud = ('cloud_cover' in output)
    has_otsu  = ('MNDWI_threshold' in output)

    if has_otsu:
        otsu_vals = np.array([output['MNDWI_threshold'][i] for i in idx_validos], dtype=float)
        otsu_med = np.nanmedian(otsu_vals)
    else:
        otsu_med = 0.0

    def score(i):
        # 1) penaliza shorelines cortas (menos puntos => peor)
        sl_len = len(output['shorelines'][i])
        pen_len = 1.0 / max(sl_len, 1)  # más grande si sl_len es pequeño

        # 2) nube (si existe): menor es mejor
        if has_cloud:
            cc = float(output['cloud_cover'][i])
        else:
            cc = 0.0

        # 3) estabilidad Otsu (si existe): distancia a mediana, menor es mejor
        if has_otsu:
            otsu = float(output['MNDWI_threshold'][i])
            d_otsu = abs(otsu - otsu_med)
        else:
            d_otsu = 0.0

        # combinación (ajustable): nube domina, luego otsu, luego longitud
        return (10.0 * cc) + (2.0 * d_otsu) + (2000.0 * pen_len)

    idx_ini = min(ventana_ini, key=score)
    idx_fin = min(ventana_fin, key=score)
    return idx_ini, idx_fin

**DESCARGA TODA LA DATABASE**

In [ ]:
""" import shutil
import os

base = "data"

for carpeta in os.listdir(base):
    ruta = os.path.join(base, carpeta, "Reporte_Ejecutivo")

    if os.path.exists(ruta):
        zip_name = f"/content/{carpeta}_Reporte.zip"
        shutil.make_archive(zip_name.replace(".zip", ""), 'zip', ruta)

        from google.colab import files
        files.download(zip_name)

"""##

**Descarga Sólo Resultados Por Segmento**

In [ ]:
import os
import zipfile

base = "data"
zip_path = "/content/segmentos_completos_filtrados.zip"

excluir = ["L8", "S2", "jpg_files"]

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for carpeta in os.listdir(base):
        ruta_base = os.path.join(base, carpeta)

        if not os.path.isdir(ruta_base):
            continue

        for root, dirs, files in os.walk(ruta_base):

            # 🔴 excluir carpetas pesadas
            dirs[:] = [d for d in dirs if d not in excluir]

            for file in files:
                ruta_completa = os.path.join(root, file)

                # mantener estructura dentro del zip
                ruta_relativa = os.path.relpath(ruta_completa, base)

                zipf.write(ruta_completa, ruta_relativa)

print("✅ ZIP único creado:", zip_path)

In [ ]:
from google.colab import files
files.download(zip_path)

NUEVOS CODES

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SCRIPT MEJORADO: Mapas de Vectores de Desplazamiento (Erosión/Crecimiento)
# Basado en CoastSat - Versión optimizada y robusta
# ══════════════════════════════════════════════════════════════════════════════

import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import Image, display
from shapely.geometry import LineString
from coastsat import SDS_tools, SDS_transects
from google.colab import drive
import warnings
warnings.filterwarnings('ignore')

# ── CONFIG ────────────────────────────────────────────────────────────────────
drive.mount('/content/drive')

filepath = '/content/drive/MyDrive/coast'

nombres_segmentos = [
    'segmento_costa_oeste_puerto',
    'segmento_malecon_general',
    'segmento_playas_salgar_principal',
    'segmento_puerto_salgar_amplio',
    'segmento_sabanilla_limite_norte',
    'segmento_transicion_salgar_sabanilla'
]

# Parámetros de transectos
espaciado_m = 300
longitud_mitad = 300
min_len_sl = 80

# Settings mejorados para QC
settings_transects = {
    'along_dist': 50,
    'min_points': 1,
    'max_std': 50,
    'max_range': 100,
    'min_chainage': -500,
    'multiple_inter': 'max',
    'auto_prc': 0.1,
}

settings_outliers = {
    'otsu_threshold': [np.nan, np.nan],
    'max_cross_change': 50,
    'plot_fig': False,
}

# Umbrales para visualización
UMBRAL_ETIQUETA_M = 100    # Solo etiqueta si cambio es >= a esto
UMBRAL_DIBUJO_M = 500      # No dibuja si cambio supera esto

# ─────────────────────────────────────────────────────────────────────────────

def shoreline_valida(sl, max_jump_factor=10):
    """Valida geometría de shoreline (sin loops ni saltos extremos)"""
    if sl is None or len(sl) < min_len_sl:
        return False
    dif = np.diff(sl, axis=0)
    step = np.sqrt(dif[:, 0]**2 + dif[:, 1]**2)
    if len(step) == 0:
        return False
    med = np.median(step)
    if med <= 0:
        return False
    return np.max(step) <= max_jump_factor * med


def calcular_estadisticas(df, col):
    """Calcula estadísticas para un transecto"""
    valido = df[['dates', col]].dropna()
    if len(valido) < 2:
        return None

    d0 = float(valido.iloc[0][col])
    d1 = float(valido.iloc[-1][col])
    cambio_neto = d1 - d0

    # Valores válidos
    valores = valido[col].values
    cambio_min = np.min(valores) - np.max(valores)
    cambio_max = np.max(valores) - np.min(valores)
    cambio_promedio = np.mean(np.abs(np.diff(valores)))

    return {
        'fecha_inicio': valido['dates'].iloc[0],
        'fecha_fin': valido['dates'].iloc[-1],
        'd0': d0,
        'd1': d1,
        'cambio_neto': cambio_neto,
        'cambio_min': cambio_min,
        'cambio_max': cambio_max,
        'cambio_promedio': cambio_promedio,
        'n_observaciones': len(valido)
    }


# ══════════════════════════════════════════════════════════════════════════════
# PROCESAMIENTO PRINCIPAL
# ══════════════════════════════════════════════════════════════════════════════

for sitename in nombres_segmentos:
    print(f"\n{'='*70}")
    print(f"📊 PROCESANDO: {sitename}")
    print(f"{'='*70}")

    carpeta = os.path.join(filepath, sitename)

    # Buscar PKL (prioridad: merged > output.pkl)
    archivo_pkl = os.path.join(carpeta, f'{sitename}_output_merged.pkl')
    if not os.path.exists(archivo_pkl):
        archivo_pkl = os.path.join(carpeta, f'{sitename}_output.pkl')

    if not os.path.exists(archivo_pkl):
        print(f"  ❌ PKL no encontrado")
        continue

    print(f"  📂 Cargando PKL...")
    with open(archivo_pkl, 'rb') as f:
        output = pickle.load(f)

    output = SDS_tools.remove_duplicates(output)
    output = SDS_tools.remove_inaccurate_georef(output, 10)

    if len(output.get('shorelines', [])) == 0:
        print(f"  ❌ Sin shorelines")
        continue

    print(f"     Shorelines: {len(output['shorelines'])}")
    print(f"     Período: {output['dates'][0].strftime('%Y-%m-%d')} → {output['dates'][-1].strftime('%Y-%m-%d')}")

    # ────────────────────────────────────────────────────────────────────────
    # 1) FILTRAR SHORELINES VÁLIDAS
    # ────────────────────────────────────────────────────────────────────────
    print(f"\n  🔍 Validando shorelines...")
    idx_validos = []
    for i, sl in enumerate(output['shorelines']):
        if shoreline_valida(sl):
            idx_validos.append(i)

    if len(idx_validos) == 0:
        print(f"  ❌ Sin shorelines válidas")
        continue

    idx_validos = sorted(idx_validos, key=lambda i: output['dates'][i])
    print(f"     Válidas: {len(idx_validos)}")

    idx_ini = idx_validos[0]
    idx_fin = idx_validos[-1]

    # ────────────────────────────────────────────────────────────────────────
    # 2) CREAR TRANSECTOS
    # ────────────────────────────────────────────────────────────────────────
    print(f"\n  📍 Creando transectos...")
    idx_base = idx_validos[len(idx_validos) // 2]
    L0 = LineString(output['shorelines'][idx_base])

    if L0.length <= 0:
        print(f"  ❌ Shoreline base sin longitud")
        continue

    transects = {}
    for i, dist in enumerate(np.arange(10, L0.length, espaciado_m)):
        p = L0.interpolate(dist)
        p_antes = L0.interpolate(max(0, dist - 5))
        p_despues = L0.interpolate(min(L0.length, dist + 5))
        t = np.array([p_despues.x - p_antes.x, p_despues.y - p_antes.y])
        nrm = np.linalg.norm(t)
        t = t / nrm if nrm > 0 else np.array([1.0, 0.0])
        n = np.array([-t[1], t[0]])
        p1 = np.array([p.x, p.y]) - n * longitud_mitad
        p2 = np.array([p.x, p.y]) + n * longitud_mitad
        transects[f'T_{i+1}'] = np.array([p1, p2])

    print(f"     ✓ {len(transects)} transectos")

    # ────────────────────────────────────────────────────────────────────────
    # 3) INTERSECCIONES + QC + OUTLIERS
    # ────────────────────────────────────────────────────────────────────────
    print(f"\n  ✓ Aplicando QC y removiendo outliers...")
    cross_distance = SDS_transects.compute_intersection_QC(output, transects, settings_transects)

    if not cross_distance:
        print(f"  ❌ Sin transectos tras QC")
        continue

    cross_distance = SDS_transects.reject_outliers(cross_distance, output, settings_outliers)

    if len(cross_distance) == 0:
        print(f"  ❌ Sin transectos tras outlier removal")
        continue

    print(f"     ✓ {len(cross_distance)} transectos válidos")

    # ────────────────────────────────────────────────────────────────────────
    # 4) CREAR DATAFRAME CON RESULTADOS
    # ────────────────────────────────────────────────────────────────────────
    df = pd.DataFrame({'dates': output['dates']})
    for k in cross_distance.keys():
        df[k] = cross_distance[k]

    # Calcular estadísticas por transecto
    stats_transectos = {}
    for col in cross_distance.keys():
        stats = calcular_estadisticas(df, col)
        if stats:
            stats_transectos[col] = stats

    # ────────────────────────────────────────────────────────────────────────
    # 5) MAPA A: VECTORES DE DESPLAZAMIENTO NETO
    # ────────────────────────────────────────────────────────────────────────
    print(f"\n  🗺️  Generando mapa de vectores...")

    fig, ax = plt.subplots(figsize=(14, 11))
    ax.axis('equal')
    ax.grid(linestyle=':', color='0.5', alpha=0.3)

    sl_inicio = output['shorelines'][idx_ini]
    sl_fin = output['shorelines'][idx_fin]
    fecha_inicio = output['dates'][idx_ini].strftime('%Y-%m-%d')
    fecha_fin = output['dates'][idx_fin].strftime('%Y-%m-%d')

    # Dibujar shorelines de inicio y fin
    ax.plot(sl_inicio[:, 0], sl_inicio[:, 1], 'b-', linewidth=2.5, label=f'Costa Inicial ({fecha_inicio})', alpha=0.7)
    ax.plot(sl_fin[:, 0], sl_fin[:, 1], 'r-', linewidth=2.5, label=f'Costa Final ({fecha_fin})', alpha=0.7)

    # Estadísticas globales
    cambios_totales = []
    cambios_acrecion = []
    cambios_erosion = []

    # Dibujar vectores de desplazamiento
    for col in cross_distance.keys():
        stats = stats_transectos.get(col)
        if not stats:
            continue

        cambio_neto = stats['cambio_neto']
        cambios_totales.append(abs(cambio_neto))

        # Filtro: outliers brutales no se dibujan
        if abs(cambio_neto) > UMBRAL_DIBUJO_M:
            continue

        p1, p2 = transects[col][0], transects[col][1]
        vec = p2 - p1
        norm = np.linalg.norm(vec)
        if norm == 0:
            continue
        u = vec / norm

        pt0 = p1 + u * stats['d0']
        pt1 = p1 + u * stats['d1']

        # Dibujar línea de desplazamiento
        color_linea = 'green' if cambio_neto > 0 else 'red'
        alpha_linea = 0.3 + (min(abs(cambio_neto), UMBRAL_ETIQUETA_M) / UMBRAL_ETIQUETA_M) * 0.7

        ax.plot([pt0[0], pt1[0]], [pt0[1], pt1[1]],
                color=color_linea, linewidth=1.5, alpha=alpha_linea, zorder=2)
        ax.plot(pt0[0], pt0[1], 'o', color=color_linea, markersize=4, alpha=0.6, zorder=3)
        ax.plot(pt1[0], pt1[1], 's', color=color_linea, markersize=5, alpha=0.8, zorder=3)

        # Etiquetas para cambios significativos
        if abs(cambio_neto) >= UMBRAL_ETIQUETA_M:
            mid = (pt0 + pt1) / 2
            signo = '+' if cambio_neto >= 0 else ''
            ax.text(mid[0], mid[1], f'{signo}{cambio_neto:.0f}m',
                    color=color_linea, fontsize=9, fontweight='bold',
                    ha='center', va='center',
                    bbox=dict(facecolor='white', alpha=0.85, edgecolor=color_linea, pad=2, linewidth=1))

        # Contabilizar cambios
        if cambio_neto > 0:
            cambios_acrecion.append(cambio_neto)
        else:
            cambios_erosion.append(cambio_neto)

    # Configurar título y leyenda
    titulo = f'Vectores de Desplazamiento Neto: {sitename}\n'
    titulo += f'Período: {fecha_inicio} → {fecha_fin}'

    if cambios_totales:
        cambio_promedio = np.mean(cambios_totales)
        cambio_maximo = np.max(cambios_totales)
        titulo += f'\nPromedio: {cambio_promedio:.1f}m | Máximo: {cambio_maximo:.1f}m'

    ax.set_title(titulo, fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel('Eastings [EPSG:32618]', fontsize=11)
    ax.set_ylabel('Northings [EPSG:32618]', fontsize=11)

    # Leyenda mejorada
    acrecion_patch = mpatches.Patch(color='green', label=f'Ganancia/Acreción (+m) - {len(cambios_acrecion)} transectos')
    erosion_patch = mpatches.Patch(color='red', label=f'Pérdida/Erosión (-m) - {len(cambios_erosion)} transectos')

    handles_lineas, labels_lineas = ax.get_legend_handles_labels()
    ax.legend(handles=handles_lineas + [acrecion_patch, erosion_patch],
              labels=labels_lineas + [acrecion_patch.get_label(), erosion_patch.get_label()],
              loc='upper left', frameon=True, fontsize=10, title='Leyenda', title_fontsize=11)

    plt.tight_layout()
    ruta_mapa = os.path.join(carpeta, f'{sitename}_Mapa_Vectores_Metros.png')
    plt.savefig(ruta_mapa, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"     ✓ Guardado: {sitename}_Mapa_Vectores_Metros.png")
    display(Image(filename=ruta_mapa))

    # ────────────────────────────────────────────────────────────────────────
    # 6) GRÁFICO B: SERIE TEMPORAL CON TENDENCIA SUAVIZADA
    # ────────────────────────────────────────────────────────────────────────
    print(f"\n  📈 Generando gráfico de tendencia...")

    if not df.drop(columns=['dates']).empty:
        # Calcular promedio de todos los transectos
        tendencia_general = df.drop(columns=['dates']).mean(axis=1)

        # Centrar en el primer valor
        if not tendencia_general.dropna().empty:
            tendencia_general = tendencia_general - tendencia_general.dropna().iloc[0]

            fechas_validas = [d for i, d in enumerate(output['dates']) if not np.isnan(tendencia_general.iloc[i])]
            tendencia_valida = tendencia_general.dropna().values

            # Calcular promedio mensual
            try:
                _, dates_month, chainage_month, _ = SDS_transects.monthly_average(fechas_validas, tendencia_valida)
            except:
                dates_month = fechas_validas
                chainage_month = tendencia_valida

            fig2, ax2 = plt.subplots(figsize=(14, 6))
            ax2.grid(linestyle=':', color='0.5', alpha=0.3)

            # Datos diarios (puntos)
            ax2.scatter(df['dates'], tendencia_general, marker='+', color='gray', s=30, alpha=0.5, label='Observaciones diarias')

            # Tendencia suavizada (línea)
            ax2.plot(dates_month, chainage_month, '-o', color='darkblue', linewidth=2.5,
                    markersize=6, label='Tendencia Suavizada (Mensual)', zorder=3)

            # Línea de referencia
            ax2.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)

            # Sombreado de regiones
            ax2.fill_between(dates_month, 0, chainage_month, where=(np.array(chainage_month) >= 0),
                            alpha=0.2, color='green', label='Crecimiento/Acreción')
            ax2.fill_between(dates_month, 0, chainage_month, where=(np.array(chainage_month) < 0),
                            alpha=0.2, color='red', label='Erosión/Pérdida')

            ax2.set_title(f'Evolución Temporal de la Línea de Costa - {sitename}\n(Promedio de todos los transectos)',
                         fontsize=13, fontweight='bold', pad=12)
            ax2.set_ylabel('Desplazamiento [Metros]\n(+ Crecimiento / - Erosión)', fontsize=11)
            ax2.set_xlabel('Fecha', fontsize=11)

            # Estadísticas finales
            delta_total = chainage_month[-1] - chainage_month[0]
            signo = '+' if delta_total >= 0 else ''
            ax2.text(0.02, 0.98, f'Cambio Total: {signo}{delta_total:.1f}m\nPeríodo: {len(fechas_validas)} observaciones',
                    transform=ax2.transAxes, fontsize=10, verticalalignment='top',
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

            ax2.legend(loc='lower left', fontsize=10, frameon=True)
            plt.xticks(rotation=45)
            plt.tight_layout()

            ruta_tendencia = os.path.join(carpeta, f'{sitename}_Tendencia_Erosion.png')
            plt.savefig(ruta_tendencia, dpi=300, bbox_inches='tight')
            plt.close(fig2)
            print(f"     ✓ Guardado: {sitename}_Tendencia_Erosion.png")
            display(Image(filename=ruta_tendencia))
        else:
            print(f"  ⚠️  Sin datos válidos para tendencia")
    else:
        print(f"  ⚠️  Transectos con demasiado ruido")

    print(f"\n  ✅ {sitename} completado")

print(f"\n\n{'='*70}")
print("✅ PROCESAMIENTO FINALIZADO")
print(f"{'='*70}")


**DEFINICIÓN COLOR CAMBIO DE COSTA**


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SCRIPT FINAL: Análisis con PKL de Análisis (histórico 2023+)
# ══════════════════════════════════════════════════════════════════════════════

import os
import json
import pickle
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import pytz
from shapely.geometry import LineString
from coastsat import SDS_tools, SDS_transects
from google.colab import files, drive

drive.mount('/content/drive')

# ── CONFIG ────────────────────────────────────────────────────────────────────
filepath    = '/content/drive/MyDrive/coast'
MIN_PUNTOS  = 2
ROLLING_DIAS = 60

espaciado_m    = 300
longitud_mitad = 300
min_len_sl     = 80

settings_transects = {
    'along_dist': 50,
    'min_points': 1,
    'max_std': 50,
    'max_range': 100,
    'min_chainage': -500,
    'multiple_inter': 'max',
    'auto_prc': 0.1,
}

settings_outliers = {
    'otsu_threshold': [np.nan, np.nan],
    'max_cross_change': 50,
    'plot_fig': False,
}

UMBRAL_DIBUJO_M = 500

MAPA_ID_SITENAME = {
    'c':                                    'segmento_malecon_general',
    'segmento_puerto_salgar_amplio':        'segmento_puerto_salgar_amplio',
    'segmento_costa_oeste_puerto':          'segmento_costa_oeste_puerto',
    'segmento_playas_salgar_principal':     'segmento_playas_salgar_principal',
    'segmento_transicion_salgar_sabanilla': 'segmento_transicion_salgar_sabanilla',
    'segmento_sabanilla_limite_norte':      'segmento_sabanilla_limite_norte',
}

# ─────────────────────────────────────────────────────────────────────────────

def shoreline_valida(sl, max_jump_factor=10):
    if sl is None or len(sl) < min_len_sl:
        return False
    dif  = np.diff(sl, axis=0)
    step = np.sqrt(dif[:, 0]**2 + dif[:, 1]**2)
    if len(step) == 0:
        return False
    med = np.median(step)
    if med <= 0:
        return False
    return np.max(step) <= max_jump_factor * med


def reject_outliers_permisive(cross_distance, output, settings):
    chain_dict = dict([])

    for i, key in enumerate(list(cross_distance.keys())):
        chainage = cross_distance[key].copy()
        if sum(np.isnan(chainage)) == len(chainage):
            continue

        idx_nonan = np.where(~np.isnan(chainage))[0]
        chainage1 = [chainage[k] for k in idx_nonan]
        dates1 = [output['dates'][k] for k in idx_nonan]

        chainage2 = chainage1
        dates2 = dates1

        if len(chainage2) < 2:
            continue

        chainage3, dates3 = identify_outliers_light(chainage2, dates2, settings['max_cross_change'])

        idx_kept = []
        for date in output['dates']:
            idx_kept.append(date in dates3)
        chainage[~np.array(idx_kept)] = np.nan
        chain_dict[key] = chainage

    return chain_dict


def identify_outliers_light(chainage, dates, cross_change):
    chainage_temp = chainage.copy()
    dates_temp = dates.copy()

    max_iterations = 5
    iteration = 0

    while iteration < max_iterations:
        iteration += 1
        found_outlier = False

        for k in range(len(chainage_temp)):

            if k == 0:
                if len(chainage_temp) > 1:
                    diff = chainage_temp[k] - chainage_temp[k+1]
                    if np.abs(diff) > 2.0 * cross_change:
                        chainage_temp.pop(k)
                        dates_temp.pop(k)
                        found_outlier = True
                        break

            elif k == len(chainage_temp) - 1:
                diff = chainage_temp[k] - chainage_temp[k-1]
                if np.abs(diff) > 2.0 * cross_change:
                    chainage_temp.pop(k)
                    dates_temp.pop(k)
                    found_outlier = True
                    break

            else:
                diff_m1 = chainage_temp[k] - chainage_temp[k-1]
                diff_p1 = chainage_temp[k] - chainage_temp[k+1]

                condition1 = np.abs(diff_m1) > 2.5 * cross_change
                condition2 = np.abs(diff_p1) > 2.5 * cross_change
                condition3 = np.sign(diff_p1) == np.sign(diff_m1)

                if np.logical_and(np.logical_and(condition1, condition2), condition3):
                    chainage_temp.pop(k)
                    dates_temp.pop(k)
                    found_outlier = True
                    break

        if not found_outlier:
            break

    return chainage_temp, dates_temp


hoy = datetime.now(pytz.UTC)

resultado = {
    'generado': hoy.strftime('%Y-%m-%d'),
    'segmentos': {}
}

print("╔════════════════════════════════════════════════════════════════╗")
print("║  ANÁLISIS CON PKL DE ANÁLISIS  (HISTÓRICO 2023+)        ║")
print("╚════════════════════════════════════════════════════════════════╝\n")

for web_id, sitename in MAPA_ID_SITENAME.items():
    print(f"\n{'='*70}")
    print(f"▶ {sitename}")
    print(f"{'='*70}")

    sitepath = os.path.join(filepath, sitename)
    analisis_abril = os.path.join(sitepath, 'Analisis Abril')

    # Buscar PKL en Análisis Abril
    if not os.path.exists(analisis_abril):
        print(f"  ❌ No existe: Analisis Abril")
        continue

    pkl_files = [f for f in os.listdir(analisis_abril) if f.endswith('_output.pkl')]

    if not pkl_files:
        print(f"  ❌ Sin PKL en Analisis Abril")
        continue

    pkl_path = os.path.join(analisis_abril, pkl_files[0])

    print(f"\n  📦 Cargando: {pkl_files[0]}")
    with open(pkl_path, 'rb') as f:
        output = pickle.load(f)

    print(f"     Shorelines: {len(output['shorelines'])}")
    print(f"     Período: {output['dates'][0].strftime('%Y-%m-%d')} → {output['dates'][-1].strftime('%Y-%m-%d')}")

    if not output.get('shorelines'):
        print(f"  ❌ Sin shorelines")
        continue

    print(f"\n  🔍 Validando shorelines...")
    idx_validos = sorted(
        [i for i, sl in enumerate(output['shorelines']) if shoreline_valida(sl)],
        key=lambda i: output['dates'][i]
    )

    print(f"     Válidas: {len(idx_validos)}")

    if len(idx_validos) < 2:
        print(f"  ❌ Menos de 2 shorelines válidas")
        continue

    print(f"\n  📍 Creando transectos...")
    idx_base = idx_validos[len(idx_validos) // 2]
    L0 = LineString(output['shorelines'][idx_base])

    if L0.length <= 0:
        print(f"  ❌ Shoreline sin longitud")
        continue

    transects = {}
    for i, dist in enumerate(np.arange(10, L0.length, espaciado_m)):
        p        = L0.interpolate(dist)
        p_antes  = L0.interpolate(max(0, dist - 5))
        p_desp   = L0.interpolate(min(L0.length, dist + 5))
        t        = np.array([p_desp.x - p_antes.x, p_desp.y - p_antes.y])
        nrm      = np.linalg.norm(t)
        t        = t / nrm if nrm > 0 else np.array([1.0, 0.0])
        n        = np.array([-t[1], t[0]])
        p1       = np.array([p.x, p.y]) - n * longitud_mitad
        p2       = np.array([p.x, p.y]) + n * longitud_mitad
        transects[f'T_{i+1}'] = np.array([p1, p2])

    print(f"     Transectos: {len(transects)}")

    print(f"\n  ✓ Aplicando QC...")
    cross = SDS_transects.compute_intersection_QC(output, transects, settings_transects)

    if not cross:
        print(f"  ❌ Sin transectos tras QC")
        continue

    print(f"\n  🧹 Removiendo outliers...")
    cross = reject_outliers_permisive(cross, output, settings_outliers)

    if not cross:
        print(f"  ❌ Sin transectos")
        continue

    print(f"\n  📊 Procesando series...")
    df = pd.DataFrame({'dates': output['dates']})
    for k, v in cross.items():
        df[k] = v

    cols_ok = []
    for k in cross:
        valido = df[['dates', k]].dropna()
        if len(valido) < 2:
            continue
        cols_ok.append(k)

    if not cols_ok:
        cols_ok = list(cross.keys())

    print(f"\n  📈 Calculando cambios...")
    X  = df[cols_ok].copy()
    lo = X.quantile(0.10, axis=1)
    hi = X.quantile(0.90, axis=1)
    for c in cols_ok:
        X[c] = X[c].where((X[c] >= lo) & (X[c] <= hi))

    serie = X.mean(axis=1, skipna=True)
    first = serie.dropna().iloc[0] if not serie.dropna().empty else 0.0
    serie = serie - first

    df_s = pd.DataFrame({'dates': df['dates'], 'val': serie.values})
    df_s = df_s.set_index('dates').sort_index()
    smooth = df_s['val'].rolling(
        f'{ROLLING_DIAS}D', min_periods=3, center=True
    ).median()

    smooth_total = smooth.dropna()

    if len(smooth_total) < MIN_PUNTOS:
        print(f"  ❌ Pocos puntos")
        continue

    delta_total = round(float(smooth_total.iloc[-1]) - float(smooth_total.iloc[0]), 2)
    signo_total = '+' if delta_total >= 0 else ''

    resultado['segmentos'][web_id] = {
        'delta_m': delta_total,
        'n_transectos': len(cols_ok),
        'n_observaciones': len(smooth_total),
        'fecha_inicio': smooth_total.index[0].strftime('%Y-%m-%d'),
        'fecha_fin': smooth_total.index[-1].strftime('%Y-%m-%d'),
        'duracion_dias': (smooth_total.index[-1] - smooth_total.index[0]).days,
    }

    print(f"\n  ✅ RESULTADO:")
    print(f"     Delta: {signo_total}{delta_total} m")
    print(f"     Período: {smooth_total.index[0].strftime('%Y-%m-%d')} → {smooth_total.index[-1].strftime('%Y-%m-%d')} ({(smooth_total.index[-1] - smooth_total.index[0]).days} días)")
    print(f"     Transectos: {len(cols_ok)}")
    print(f"     Observaciones: {len(smooth_total)}")

print(f"\n\n{'='*70}")
print("📁 GUARDANDO RESULTADOS")
print(f"{'='*70}")

json_str  = json.dumps(resultado, ensure_ascii=False, indent=2)
json_path = '/content/resultados_web.json'

with open(json_path, 'w', encoding='utf-8') as fj:
    fj.write(json_str)

drive_json = os.path.join(filepath, 'resultados_web.json')
with open(drive_json, 'w', encoding='utf-8') as fj:
    fj.write(json_str)

print(f'\n✅ resultados_web.json guardado')
print(f'\n{json_str}')

print(f"\n📊 RESUMEN:")
print(f"   Segmentos: {len(resultado['segmentos'])}/{len(MAPA_ID_SITENAME)}")
for web_id, data in resultado['segmentos'].items():
    signo = '+' if data['delta_m'] >= 0 else ''
    print(f"   • {web_id}: {signo}{data['delta_m']} m")

try:
    files.download(json_path)
except:
    pass
